# TIMIT × self-supervised speech models: one utterance, end to end

One utterance — **`TRAIN/DR1/FCJF0/SA1`** (*"She had your dark suit in greasy wash water all year."*) —
worked through independently on each of five models:

| # | Model | HF checkpoint |
|---|-------|---------------|
| A | wav2vec 2.0 base | `facebook/wav2vec2-base` |
| B | WavLM base | `microsoft/wavlm-base` |
| C | WavLM base+ | `microsoft/wavlm-base-plus` |
| D | HuBERT base | `facebook/hubert-base-ls960` |
| E | data2vec audio base | `facebook/data2vec-audio-base` |

Every model section contains the full set of three tasks —
**1. Read one utterance (看懂一条数据) / 2. Sample-to-model-frame alignment (完成对齐) /
3. Probe the frozen model's layers (探索各层表示)** — and the three are independent of one another.

> **How to run**: execute **Section 0** first (it defines every shared helper); after that any
> model section runs on its own.
> Task 1 is model-independent, so its output is identical in all five sections — it is repeated
> so that each model section stands alone.

## 0. Setup and shared helpers (**run this section first**)

### 0.1 Imports and global configuration

In [ ]:
import os, warnings, logging, copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import soundfile as sf
import matplotlib.pyplot as plt

from IPython.display import Audio, display
import transformers
from transformers import AutoConfig, AutoModel, AutoFeatureExtractor

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

torch.manual_seed(0)
np.random.seed(0)

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 170)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = False

# ---- the utterance ----
UTT_STEM = "/Users/agrfhyl/Documents/timit/TIMIT/TRAIN/DR1/FCJF0/SA1"
SR = 16000

# ---- the five models ----
MODELS = {
    "wav2vec2-base":       "facebook/wav2vec2-base",
    "wavlm-base":          "microsoft/wavlm-base",
    "wavlm-base-plus":     "microsoft/wavlm-base-plus",
    "hubert-base":         "facebook/hubert-base-ls960",
    "data2vec-audio-base": "facebook/data2vec-audio-base",
}

print("torch", torch.__version__, "| transformers", transformers.__version__)
for ext in (".WAV", ".TXT", ".WRD", ".PHN"):
    p = UTT_STEM + ext
    print(f"  {ext:5s} {'OK     ' if os.path.exists(p) else 'MISSING'} {p}")

### 0.2 Reading SPHERE audio

TIMIT's `.WAV` files are **not** RIFF/WAVE — they are **NIST SPHERE**: a 1024-byte ASCII header
followed by raw PCM. `libsndfile` (and therefore `soundfile`) supports the format natively.
The manual fallback below is here both as a safety net for `libsndfile` builds without NIST
support, and so the header fields are actually read rather than skipped over.

In [ ]:
def read_sphere_header(path):
    """Parse a NIST SPHERE header -> (magic, header_bytes, fields dict)."""
    fields = {}
    with open(path, "rb") as f:
        magic = f.readline().decode("ascii").strip()               # 'NIST_1A'
        header_bytes = int(f.readline().decode("ascii").strip())   # 1024
        while True:
            line = f.readline()
            if not line:
                break
            line = line.decode("ascii").rstrip("\n")
            if line.strip() == "end_head":
                break
            parts = line.split()
            if len(parts) < 3:
                continue
            name, typ, val = parts[0], parts[1], " ".join(parts[2:])
            if typ.startswith("-i"):
                val = int(val)
            elif typ.startswith("-r"):
                val = float(val)
            fields[name] = val
    return magic, header_bytes, fields


def read_sphere(path):
    """Read a TIMIT SPHERE .WAV -> (float32 waveform in [-1, 1], sample_rate, how)."""
    try:
        x, sr = sf.read(path, dtype="float32", always_2d=False)
        return np.asarray(x, dtype=np.float32), int(sr), "soundfile (libsndfile NIST)"
    except Exception:
        magic, header_bytes, F = read_sphere_header(path)
        assert magic.startswith("NIST"), f"not a SPHERE file: {magic}"
        coding = str(F.get("sample_coding", "pcm"))
        assert coding.startswith("pcm"), f"SPHERE is compressed ({coding}); needs sph2pipe"
        n, nb = F["sample_count"], F["sample_n_bytes"]
        dtype = "<i2" if str(F.get("sample_byte_format", "01")) == "01" else ">i2"
        with open(path, "rb") as f:
            f.seek(header_bytes)
            raw = f.read(n * nb)
        x = np.frombuffer(raw, dtype=dtype).astype(np.float32) / 32768.0
        return x, int(F["sample_rate"]), "manual SPHERE parse"

### 0.3 Reading `.TXT` / `.WRD` / `.PHN`

All three annotation files index the waveform in **sample numbers** (not seconds, not
milliseconds) and describe **half-open intervals `[start, end)`** at 16 kHz.

In [ ]:
def read_timit_txt(path):
    """.TXT -> (start_sample, end_sample, sentence transcript)"""
    line = open(path).read().strip()
    s, e, txt = line.split(maxsplit=2)
    return int(s), int(e), txt


def read_timit_seg(path, sr=SR):
    """.PHN / .WRD -> DataFrame[start, end, label, t_start, t_end, dur_ms]"""
    rows = []
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        s, e, lab = line.split(maxsplit=2)
        rows.append((int(s), int(e), lab))
    df = pd.DataFrame(rows, columns=["start", "end", "label"])
    df["t_start"] = df["start"] / sr
    df["t_end"]   = df["end"]   / sr
    df["dur_ms"]  = (df["end"] - df["start"]) / sr * 1000
    return df


# TIMIT-61 -> broad manner classes (used only for colouring and separability stats)
_CLASS_MAP = {
    "vowel":     "iy ih eh ey ae aa aw ay ah ao oy ow uh uw ux er ax ix axr ax-h".split(),
    "semivowel": "l r w y hh hv el".split(),
    "nasal":     "m n ng em en eng nx".split(),
    "fricative": "s sh z zh f th v dh".split(),
    "affricate": "jh ch".split(),
    "stop":      "b d g p t k dx q".split(),
    "closure":   "bcl dcl gcl pcl tcl kcl".split(),
    "silence":   "h# pau epi".split(),
}
PHONE2CLASS = {p: c for c, ps in _CLASS_MAP.items() for p in ps}
CLASS_COLOR = {
    "vowel": "#4C78A8", "semivowel": "#72B7B2", "nasal": "#54A24B",
    "fricative": "#EECA3B", "affricate": "#F58518", "stop": "#E45756",
    "closure": "#B279A2", "silence": "#9C9C9C", "other": "#333333",
}

def phone_class(p):
    return PHONE2CLASS.get(p, "other")

### 0.4 Plotting the waveform with phone / word annotation overlaid

In [ ]:
def plot_wave_with_phones(x, phones, words=None, sr=SR, t0=None, t1=None,
                          title="", ax=None, label_every=1):
    """Waveform + PHN boundaries and labels (above), optional WRD tier (below).
    Colour encodes the broad phone class."""
    if ax is None:
        _, ax = plt.subplots(figsize=(16, 4))
    t  = np.arange(len(x)) / sr
    lo = 0.0 if t0 is None else t0
    hi = len(x) / sr if t1 is None else t1
    m  = (t >= lo) & (t <= hi)
    ax.plot(t[m], x[m], lw=0.6, color="#333333", zorder=3)

    amp = float(np.abs(x[m]).max()) if m.any() else 1.0
    ax.set_ylim(-amp * 1.60, amp * 1.60)

    sub = phones[(phones.t_end > lo) & (phones.t_start < hi)]
    for i, r in enumerate(sub.itertuples()):
        c = CLASS_COLOR[phone_class(r.label)]
        ax.axvspan(r.t_start, r.t_end, color=c, alpha=0.16, lw=0, zorder=0)
        ax.axvline(r.t_start, color=c, lw=0.9, alpha=0.85, zorder=2)
        if i % label_every == 0:
            y = amp * (1.18 if i % 2 == 0 else 1.40)
            ax.text((max(r.t_start, lo) + min(r.t_end, hi)) / 2, y, r.label,
                    ha="center", va="center", fontsize=8.5, color=c,
                    weight="bold", zorder=4)
    if len(sub):
        ax.axvline(sub.iloc[-1].t_end, color="#666666", lw=0.9, alpha=0.85, zorder=2)

    if words is not None:
        wsub = words[(words.t_end > lo) & (words.t_start < hi)]
        for j, r in enumerate(wsub.itertuples()):
            y = -amp * (1.16 if j % 2 == 0 else 1.40)
            ax.hlines(y, r.t_start, r.t_end, color="#1f4e79", lw=2.4, zorder=4)
            ax.text((r.t_start + r.t_end) / 2, y - amp * 0.10, r.label,
                    ha="center", va="top", fontsize=9, color="#1f4e79", zorder=4)

    ax.set_xlim(lo, hi)
    ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude")
    ax.set_title(title); ax.grid(alpha=0.15)
    return ax


def legend_classes(ax):
    """Phone-class colour key, placed above the axes (title is nudged up to make room)."""
    from matplotlib.patches import Patch
    ax.set_title(ax.get_title(), pad=26)
    ax.legend(handles=[Patch(facecolor=CLASS_COLOR[c], label=c)
                       for c in ["vowel", "semivowel", "nasal", "fricative",
                                 "affricate", "stop", "closure", "silence"]],
              ncol=8, fontsize=7.5, loc="lower center",
              bbox_to_anchor=(0.5, 1.005), frameon=False)

### 0.5 The sample ↔ frame alignment arithmetic

All five models use the **identical convolutional feature extractor**:

```
kernel = [10, 3, 3, 3, 3, 2, 2] (in samples)
stride = [ 5, 2, 2, 2, 2, 2, 2]
```

* none of the convolutions pad, so the length recursion is `L ← (L − k) // s + 1`;
* **total stride** = 5·2·2·2·2·2·2 = **320** samples = **20 ms** (the frame shift);
* **receptive field**, folded back with `R ← (R − 1)·s + k` = **400** samples = **25 ms** (the frame length).

So frame `t` is computed from samples **`[320·t, 320·t + 400)`**, centred at **`320·t + 200`**.

In [ ]:
CONV_KERNEL = [10, 3, 3, 3, 3, 2, 2]
CONV_STRIDE = [5, 2, 2, 2, 2, 2, 2]


def conv_out_len(n, kernels=CONV_KERNEL, strides=CONV_STRIDE):
    """Output length of the unpadded conv stack."""
    for k, s in zip(kernels, strides):
        n = (n - k) // s + 1
    return int(n)


def total_stride(strides=CONV_STRIDE):
    out = 1
    for s in strides:
        out *= s
    return int(out)


def receptive_field(kernels=CONV_KERNEL, strides=CONV_STRIDE):
    r = 1
    for k, s in zip(reversed(kernels), reversed(strides)):
        r = (r - 1) * s + k
    return int(r)


HOP = total_stride()      # 320 samples = 20 ms
WIN = receptive_field()   # 400 samples = 25 ms


def frame_span(t, hop=HOP, win=WIN):
    """Half-open sample range that frame t is computed from."""
    return hop * t, hop * t + win


def frame_center(t, hop=HOP, win=WIN):
    return hop * t + win // 2


def frames_overlapping(start, end, T, hop=HOP, win=WIN):
    """Frames whose receptive field intersects [start, end) — permissive; boundary
    frames end up shared between neighbouring phones."""
    return [t for t in range(T)
            if frame_span(t, hop, win)[0] < end and frame_span(t, hop, win)[1] > start]


def frames_by_center(start, end, T, hop=HOP, win=WIN):
    """Frames whose receptive-field **centre** lands inside [start, end) — strict, and a
    true partition of the frames (each frame belongs to exactly one phone)."""
    return [t for t in range(T) if start <= frame_center(t, hop, win) < end]


def frame_label_table(T, phones, hop=HOP, win=WIN):
    """Per-frame table: receptive field, centre sample/time, the phone containing the
    centre, and its broad class."""
    rows = []
    for t in range(T):
        a, b = frame_span(t, hop, win)
        c = frame_center(t, hop, win)
        hit = phones[(phones.start <= c) & (phones.end > c)]
        lab = hit.iloc[0].label if len(hit) else "<none>"
        rows.append((t, a, b, c, round(c / SR, 5), lab, phone_class(lab)))
    return pd.DataFrame(rows, columns=["frame", "rf_start", "rf_end", "center",
                                       "center_s", "phone", "class"])


print(f"HOP = {HOP} samples = {HOP/SR*1000:.1f} ms   |   WIN = {WIN} samples = {WIN/SR*1000:.1f} ms")

### 0.6 Loading models / frozen forward pass

**Key point: always use the checkpoint's own `AutoFeatureExtractor` — never hand-roll the
normalisation.** These five checkpoints do *not* agree on preprocessing (`do_normalize` and
`return_attention_mask` both differ), and hand-rolling silently feeds the model the wrong
input distribution.

In [ ]:
_MODEL_CACHE = {}


def load_model(hf_id):
    """(feature_extractor, model, config), cached."""
    if hf_id not in _MODEL_CACHE:
        cfg   = AutoConfig.from_pretrained(hf_id)
        fe    = AutoFeatureExtractor.from_pretrained(hf_id)
        model = AutoModel.from_pretrained(hf_id)
        model.eval()
        _MODEL_CACHE[hf_id] = (fe, model, cfg)
    return _MODEL_CACHE[hf_id]


def forward_frozen(hf_id, x, sr=SR):
    """eval() + requires_grad=False + no_grad forward. Returns everything the later steps need."""
    fe, model, cfg = load_model(hf_id)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    inputs = fe(x, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        conv = model.feature_extractor(inputs["input_values"])   # (B, 512, T) conv output
        out  = model(**inputs, output_hidden_states=True, return_dict=True)

    return dict(fe=fe, model=model, config=cfg, inputs=inputs, conv=conv, out=out,
                hidden_states=out.hidden_states, T=out.last_hidden_state.shape[1])


def describe_frontend(hf_id):
    fe, model, cfg = load_model(hf_id)
    return pd.Series({
        "checkpoint":            hf_id,
        "model class":           type(model).__name__,
        "n params (M)":          round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
        "hidden_size":           cfg.hidden_size,
        "num_hidden_layers":     cfg.num_hidden_layers,
        "conv_kernel":           str(cfg.conv_kernel),
        "conv_stride":           str(cfg.conv_stride),
        "feat_extract_norm":     getattr(cfg, "feat_extract_norm", "(layer, fixed)"),
        "FE.do_normalize":       fe.do_normalize,
        "FE.return_attention_mask": fe.return_attention_mask,
        "FE.sampling_rate":      fe.sampling_rate,
    }, name="value").to_frame()

### 0.7 Helpers for probing the layers

In [ ]:
def layer_stats(hidden_states):
    """Per-layer mean / std / mean L2 norm of the per-frame vectors."""
    rows = []
    for i, h in enumerate(hidden_states):
        v = h[0].float()
        rows.append((i, tuple(h.shape), v.mean().item(), v.std().item(),
                     v.norm(dim=-1).mean().item()))
    return pd.DataFrame(rows, columns=["layer", "shape", "mean", "std", "mean_row_norm"])


def layer_cosine(hidden_states):
    """Layer-vs-layer: mean-centre and unit-normalise each, then average the per-frame cosine."""
    H = [Fn.normalize(h[0].float() - h[0].float().mean(0, keepdim=True), dim=-1)
         for h in hidden_states]
    L = len(H)
    M = np.zeros((L, L))
    for i in range(L):
        for j in range(L):
            M[i, j] = (H[i] * H[j]).sum(-1).mean().item()
    return M


def linear_cka(X, Y):
    """Linear CKA (rows = frames). Standard representation-similarity measure; invariant to
    rotation and isotropic scaling."""
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    return (((X.T @ Y).norm() ** 2) / ((X.T @ X).norm() * (Y.T @ Y).norm())).item()


def layer_cka(hidden_states):
    H = [h[0].float() for h in hidden_states]
    L = len(H)
    M = np.zeros((L, L))
    for i in range(L):
        for j in range(i, L):
            M[i, j] = M[j, i] = linear_cka(H[i], H[j])
    return M


def boundary_contrast(hidden_states, frame_phones):
    """Cosine similarity of adjacent frames: **within a phone** vs **across a phone boundary**.

    One of the few measures with real statistical power on a single utterance (~144 adjacent
    frame pairs), and it doubles as a check that the PHN alignment is right: the gap must be
    positive.
    """
    from sklearn.metrics import roc_auc_score
    lab   = np.asarray(frame_phones)
    cross = np.array([lab[t] != lab[t + 1] for t in range(len(lab) - 1)])
    rows = []
    for i, h in enumerate(hidden_states):
        v = Fn.normalize(h[0].float(), dim=-1)
        s = (v[:-1] * v[1:]).sum(-1).numpy()
        rows.append((i, s[~cross].mean(), s[cross].mean(),
                     s[~cross].mean() - s[cross].mean(),
                     roc_auc_score(cross, -s)))
    return pd.DataFrame(rows, columns=["layer", "within_phone", "across_boundary",
                                       "gap", "boundary_AUC"])


def phone_separability(hidden_states, frame_classes, min_count=5, seed=0):
    """Per-layer separability of broad phone classes: silhouette (unsupervised geometry)
    plus a 5-fold linear probe (supervised)."""
    from sklearn.metrics import silhouette_score
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline

    y    = np.asarray(frame_classes)
    keep = np.isin(y, [c for c in np.unique(y) if (y == c).sum() >= min_count])
    yk   = y[keep]
    cv   = StratifiedKFold(5, shuffle=True, random_state=seed)
    rows = []
    for i, h in enumerate(hidden_states):
        X = h[0].float().numpy()[keep]
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
        rows.append((i, silhouette_score(X, yk, metric="cosine"),
                     cross_val_score(clf, X, yk, cv=cv).mean()))
    return (pd.DataFrame(rows, columns=["layer", "silhouette_cos", "probe_acc_5fold"]),
            keep, yk)

### 0.8 Front-end differences across the five models (run once)

The table below is the set of facts the rest of the notebook leans on. Note that
`do_normalize` and `return_attention_mask` are **not** consistent across the five checkpoints.

In [ ]:
rows = []
for key, hf in MODELS.items():
    s = describe_frontend(hf)["value"]
    s.name = key
    rows.append(s)
front = pd.DataFrame(rows)
display(front[["checkpoint", "model class", "n params (M)", "hidden_size",
               "num_hidden_layers", "feat_extract_norm",
               "FE.do_normalize", "FE.return_attention_mask"]])

print("\nThe convolutional geometry is identical for all five:")
print(f"  conv_kernel = {CONV_KERNEL}\n  conv_stride = {CONV_STRIDE}")
print(f"  -> hop {HOP} samples ({HOP/SR*1000:.0f} ms), window {WIN} samples ({WIN/SR*1000:.0f} ms)")

---

# A. wav2vec 2.0 base — `facebook/wav2vec2-base`

`facebook/wav2vec2-base` is the **self-supervised checkpoint with no ASR fine-tuning**
(LibriSpeech 960h).

* Loading it with `AutoModel` gives a `Wav2Vec2Model` (encoder only), so the contrastive
  pre-training heads (`quantizer.*`, `project_q.*`, `project_hid.*`) are reported as
  **UNEXPECTED / discarded** weights. That is expected — they are only needed to *run*
  wav2vec 2.0 pre-training.
* Front end: `do_normalize=True` but `return_attention_mask=False` (as for the whole
  `feat_extract_norm="group"` base family).
* The first conv layer is a **GroupNorm**, whose statistics run along the time axis — see the
  empirical check in A.2.4.

### A.1 Read one utterance — 看懂一条数据

> This task is model-independent (pure TIMIT parsing), so its output is the same in all five
> model sections. It is repeated so that the **wav2vec 2.0 base** section stands on its own.

#### (A.1.1) Read the SPHERE audio

Print the 1024-byte NIST header field by field first, then read the waveform and cross-check
it against a hand-rolled parse.

In [ ]:
MODEL_KEY, HF_ID = "wav2vec2-base", "facebook/wav2vec2-base"
print(f"### wav2vec 2.0 base  (facebook/wav2vec2-base) ###\n")

magic, header_bytes, F = read_sphere_header(UTT_STEM + ".WAV")
print(f"SPHERE magic = {magic}   header = {header_bytes} bytes")
for k, v in F.items():
    print(f"   {k:20s} {v}")

x, sr, how = read_sphere(UTT_STEM + ".WAV")
print(f"\nread via: {how}")
print(f"waveform: shape={x.shape}  dtype={x.dtype}  sr={sr}  "
      f"duration={len(x)/sr:.4f} s  range=[{x.min():.4f}, {x.max():.4f}]")

# Cross-check: manual header parse + raw PCM should match libsndfile exactly
raw = np.fromfile(UTT_STEM + ".WAV", dtype="<i2",
                  offset=header_bytes, count=F["sample_count"]).astype(np.float32) / 32768.0
print(f"manual parse == soundfile ? {np.allclose(raw, x)}   (max |diff| = {np.abs(raw-x).max():.2e})")
assert sr == 16000 and x.ndim == 1

#### (A.1.2) Read `.TXT` / `.WRD` / `.PHN`

All three are **sample indices over half-open intervals `[start, end)`**.

In [ ]:
txt_s, txt_e, transcript = read_timit_txt(UTT_STEM + ".TXT")
words  = read_timit_seg(UTT_STEM + ".WRD")
phones = read_timit_seg(UTT_STEM + ".PHN")

print(f".TXT  [{txt_s}, {txt_e})  {transcript!r}")
print(f".WRD  {len(words)} words      .PHN  {len(phones)} phones\n")

display(words)
display(phones)

print("phone classes present:", sorted(set(phones.label.map(phone_class))))

# --- three real TIMIT gotchas visible in this very utterance ---
ov = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i].end - words.iloc[i+1].start)
      for i in range(len(words)-1) if words.iloc[i].end > words.iloc[i+1].start]
gp = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i+1].start - words.iloc[i].end)
      for i in range(len(words)-1) if words.iloc[i].end < words.iloc[i+1].start]
print(f"\n[gotcha 1] word spans that OVERLAP (overlap in samples): {ov}")
print(f"[gotcha 2] GAPS between words (gap in samples): {gp}")
print(f"[gotcha 3] PHN covers up to {phones.end.iloc[-1]}, but the waveform has {len(x)} "
      f"samples -> the last {len(x)-phones.end.iloc[-1]} samples are unlabelled")
print("\nBy contrast PHN itself is GAPLESS (every end == the next start):",
      bool((phones.end.values[:-1] == phones.start.values[1:]).all()))

#### (A.1.3) Play the whole utterance

In [ ]:
print(transcript)
display(Audio(x, rate=SR))

#### (A.1.4) Plot the waveform with phone boundaries and labels overlaid

Phone tier on top (colour = broad phone class), word tier in blue below. The second panel
zooms into `she had your`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8.5))
plot_wave_with_phones(x, phones, words, ax=axes[0],
                      title=f"wav2vec 2.0 base | {os.path.basename(UTT_STEM)} — {transcript}")
legend_classes(axes[0])
plot_wave_with_phones(x, phones, words, ax=axes[1], t0=0.15, t1=1.10,
                      title="zoom: 0.15 - 1.10 s  (she had your)")
fig.tight_layout()
plt.show()

#### (A.1.5) Crop a phone and listen to just that segment

Because `.PHN`'s `[start, end)` *is* a pair of array indices, cropping is literally
`x[start:end]`. Below: each phone on its own first, then the same phones with **20 ms of
context** added — many phones are near-unrecognisable in isolation, closures and stops
especially.

In [ ]:
def crop_phone(i, pad_ms=0.0):
    """Crop audio by PHN row index; pad_ms adds that much context on each side."""
    r   = phones.loc[i]
    pad = int(pad_ms / 1000 * SR)
    a, b = max(0, r.start - pad), min(len(x), r.end + pad)
    return x[a:b], r


def play_phone(i, pad_ms=0.0):
    seg, r = crop_phone(i, pad_ms)
    print(f"[{i:2d}] {r.label:4s} ({phone_class(r.label):9s})  "
          f"[{r.start}, {r.end})  {r.dur_ms:6.1f} ms  "
          f"{'+' + str(int(pad_ms)) + 'ms ctx' if pad_ms else ''}")
    display(Audio(seg, rate=SR))


# a representative spread: fricative / vowel / stop closure / stop release
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]))

print("\n---- same phones, with 20 ms of context on each side ----")
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]), pad_ms=20)

print("\n---- cropping by word works the same way ----")
for wi in [6, 7]:
    w = words.loc[wi]
    print(f"word {w.label!r}  [{w.start}, {w.end})  {w.dur_ms:.0f} ms")
    display(Audio(x[w.start:w.end], rate=SR))

With `ipywidgets` installed (`pip install ipywidgets`) the next cell gives a **dropdown**
that plays the selected phone; without it, it falls back to printing a table of choices.

In [ ]:
try:
    import ipywidgets as W

    opts = [(f"{i:2d}  {r.label:5s} {r.dur_ms:6.1f} ms  [{r.start},{r.end})", i)
            for i, r in phones.iterrows()]
    dd  = W.Dropdown(options=opts, description="phone:", layout=W.Layout(width="420px"))
    pad = W.IntSlider(value=0, min=0, max=100, step=10, description="ctx (ms):")
    outw = W.Output()

    def _on(_=None):
        with outw:
            outw.clear_output()
            play_phone(dd.value, pad_ms=pad.value)

    dd.observe(_on, names="value"); pad.observe(_on, names="value")
    display(W.VBox([W.HBox([dd, pad]), outw])); _on()
except ImportError:
    print("ipywidgets not installed -- pick manually with play_phone(i). Available phones:\n")
    print(phones.assign(cls=phones.label.map(phone_class))
                [["label", "cls", "start", "end", "dur_ms"]].to_string())
    print("\ne.g.  play_phone(13, pad_ms=20)")

### A.2 Sample-to-model-frame alignment — 完成对齐

Goal: translate `.PHN`'s `[start_sample, end_sample)` into exact encoder frame indices for
**wav2vec 2.0 base**.

#### (A.2.1) Feed in one 16 kHz waveform

Preprocess with the checkpoint's **own** `AutoFeatureExtractor` — the `do_normalize` and
`return_attention_mask` policies differ per checkpoint, and hand-rolled normalisation fails
silently.

In [ ]:
MODEL_KEY, HF_ID = "wav2vec2-base", "facebook/wav2vec2-base"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")
words    = read_timit_seg(UTT_STEM + ".WRD")
assert sr == 16000, f"model expects 16 kHz, got {sr}"

display(describe_frontend(HF_ID))

fe, model, cfg = load_model(HF_ID)
inputs = fe(x, sampling_rate=SR, return_tensors="pt")
print("feature extractor output:", {k: tuple(v.shape) for k, v in inputs.items()})
print(f"raw waveform      mean={x.mean():+.5f}  std={x.std():.5f}")
iv = inputs["input_values"][0].numpy()
print(f"fed to the model  mean={iv.mean():+.5f}  std={iv.std():.5f}   "
      f"(do_normalize={fe.do_normalize})")

#### (A.2.2) Print the conv output and every hidden-state shape

`model.feature_extractor` *is* the 7-layer conv stack; it returns `(B, 512, T)` — note that
it is **channel-first**. `output_hidden_states=True` yields **13** tensors:
`hidden_states[0]` is the input to the Transformer, `hidden_states[i]` is the output of
Transformer layer `i`, and `hidden_states[12] is last_hidden_state`.

In [ ]:
R  = forward_frozen(HF_ID, x)
hs = R["hidden_states"]
T  = R["T"]

print(f"waveform                       {tuple(R['inputs']['input_values'].shape)}   ({len(x)} samples)")
print(f"conv feature_extractor output  {tuple(R['conv'].shape)}   (B, C=512, T) channel-first")
print(f"conv transposed                {tuple(R['conv'].transpose(1,2).shape)}   (B, T, C)")
if getattr(R["out"], "extract_features", None) is not None:
    print(f"outputs.extract_features       {tuple(R['out'].extract_features.shape)}   (layer-normed conv features)")
print(f"last_hidden_state              {tuple(R['out'].last_hidden_state.shape)}")
print(f"\nhidden_states: {len(hs)} tensors (= 1 + num_hidden_layers = 1 + {cfg.num_hidden_layers})")
for i, h in enumerate(hs):
    tag = "after conv projection / before Transformer" if i == 0 else f"output of Transformer layer {i}"
    star = "   <- last_hidden_state" if i == len(hs) - 1 else ""
    print(f"   hidden_states[{i:2d}]  {tuple(h.shape)}   {tag}{star}")
print("\nhidden_states[-1] is last_hidden_state ?",
      torch.equal(hs[-1], R["out"].last_hidden_state))

# length: hand-derived formula == HF's internal helper == the actual output
n = len(x)
print(f"\nlength check  n={n} samples")
print(f"   hand-derived conv_out_len(n)               = {conv_out_len(n)}")
print(f"   HF _get_feat_extract_output_lengths(n)     = {int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])}")
print(f"   actual T                                   = {T}")
assert conv_out_len(n) == T == int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])
print(f"\nLast frame's receptive field ends at {HOP*(T-1)+WIN}; the waveform is {n} samples "
      f"-> the final {n-(HOP*(T-1)+WIN)} samples are dropped, since the convs do not pad")

#### (A.2.3) Map PHN's `[start_sample, end_sample)` onto encoder frames

Two mappings, for two different purposes:

* **overlap** — the receptive field intersects the phone. Permissive; frames get shared
  between phones. Use it to ask *"which frames ever saw this phone?"*
* **centre** — the receptive-field **centre** falls inside the phone. Strict, and it forms a
  **partition** of the frames (each frame belongs to exactly one phone). This is the one you
  must use to build per-frame classification labels.

In [ ]:
rows = []
for r in phones.itertuples():
    fo = frames_overlapping(r.start, r.end, T)
    fc = frames_by_center(r.start, r.end, T)
    rows.append((r.Index, r.label, phone_class(r.label), r.start, r.end, round(r.dur_ms, 1),
                 f"[{fo[0]},{fo[-1]}]" if fo else "-", len(fo),
                 f"[{fc[0]},{fc[-1]}]" if fc else "-", len(fc)))
align = pd.DataFrame(rows, columns=["i", "phone", "class", "start", "end", "dur_ms",
                                    "overlap_frames", "n_overlap",
                                    "center_frames", "n_center"])
display(align)

print(f"centre-mapped frames sum to {align.n_center.sum()};  T = {T}   "
      f"-> exactly a partition: {align.n_center.sum() == T}")
print(f"overlap-mapped frames sum to {align.n_overlap.sum()} (> T, boundary frames double-counted)")

lost = align[align.n_center == 0]
print(f"\nPhones that receive NO centre frame at all "
      f"(anything shorter than the 20 ms hop can vanish entirely):")
display(lost[["i", "phone", "class", "start", "end", "dur_ms", "overlap_frames"]])
print("This is the intrinsic cost of a 20 ms frame shift: in per-frame phone labels, "
      "these phones simply disappear.")

#### (A.2.4) Check whether a frame's centre really falls inside the target phone

Taking the `/s/` in `suit`, walk every overlap frame and print its receptive field, its
centre, whether that centre is inside the phone, and how much of the receptive field the
phone actually covers.

In [ ]:
TARGET = "s"                                    # try any other phone here
ti = int(align[align.phone == TARGET].i.iloc[0])
tr = phones.loc[ti]
print(f"target phone: [{ti}] {tr.label}  [{tr.start}, {tr.end})  {tr.dur_ms:.1f} ms "
      f"= [{tr.t_start:.4f}, {tr.t_end:.4f}] s\n")

chk = []
for t in frames_overlapping(tr.start, tr.end, T):
    a, b = frame_span(t)
    c    = frame_center(t)
    inside = bool(tr.start <= c < tr.end)
    ov   = max(0, min(b, tr.end) - max(a, tr.start))
    chk.append((t, a, b, c, round(c / SR, 4), inside, ov, round(100 * ov / WIN, 1)))
chk = pd.DataFrame(chk, columns=["frame", "rf_start", "rf_end", "center", "center_s",
                                 "center_inside_phone", "overlap_samples", "overlap_%"])
display(chk)
print("Look at the first and last rows: their receptive fields do intersect /s/, but their "
      "centres already lie outside it, so the centre mapping excludes them. That is "
      "precisely where the two mappings part ways.")

# the global per-frame label table
ftab = frame_label_table(T, phones)
display(ftab.head(10))
print("...")
display(ftab.tail(5))
assert (ftab.phone != "<none>").all(), "some frame centre falls outside PHN coverage"

**Empirical check.** The `[320t, 320t+400)` span above came out of pure convolution
arithmetic. Is it actually true? Replace the normalisation layers in the conv stack with
`Identity`, perturb **a single sample**, and see which frames change.

In [ ]:
probe_sample = 2440                       # an arbitrary sample
pred = [t for t in range(T) if frame_span(t)[0] <= probe_sample < frame_span(t)[1]]

# (a) normalisation removed -> pure conv arithmetic, should match the formula exactly
m2, n_repl = copy.deepcopy(model), 0
for mod in m2.feature_extractor.modules():
    for cn, ch in list(mod.named_children()):
        if isinstance(ch, (nn.GroupNorm, nn.LayerNorm, nn.BatchNorm1d)):
            setattr(mod, cn, nn.Identity()); n_repl += 1

xb = torch.randn(1, len(x)) * 0.05
xp = xb.clone(); xp[0, probe_sample] += 1.0
with torch.no_grad():
    d_pure = (m2.feature_extractor(xp) - m2.feature_extractor(xb)).abs().sum(1)[0]
got = torch.nonzero(d_pure > 1e-5).flatten().tolist()
print(f"replaced {n_repl} normalisation layer(s) in the conv stack")
print(f"perturbing sample {probe_sample} -> frames affected {got}   formula predicts {pred}   match: {got == pred}")

# (b) normalisation kept -> measure how much global information it leaks
with torch.no_grad():
    d_real = (model.feature_extractor(xp) - model.feature_extractor(xb)).abs().sum(1)[0]
mask = torch.ones(T, dtype=bool); mask[pred] = False
inside, outside = d_real[pred].sum().item(), d_real[mask].sum().item()
print(f"\nwith normalisation kept: change inside the receptive field {inside:.4f} / "
      f"outside {outside:.4f} = {inside/max(outside,1e-12):.1f}x")
print(f"normalisation in the conv stack: {[type(m).__name__ for m in model.feature_extractor.modules() if isinstance(m,(nn.GroupNorm,nn.LayerNorm))]}")
print("""
Takeaways:
  * Under pure convolution arithmetic the receptive field [320t, 320t+400) is EXACT --
    it matches for all five models.
  * But GroupNorm computes its statistics along the TIME AXIS, so every frame ends up
    dependent on the whole utterance. Note how small the ratio above is: for the four
    GroupNorm checkpoints it lands around 0.5x-1.5x, i.e. the normalisation pathway carries
    as much of the perturbation as the local receptive field does. data2vec instead uses
    per-frame LayerNorm (7 of them) and does not leak at all -- its ratio is ~1e13.
  * More important still: all of this holds only for the **conv output / hidden_states[0]**.
    Past the first self-attention block, every frame of hidden_states[i>=1] can in principle
    see the entire utterance -- the frame-to-phone alignment is a **positional**
    correspondence, not a bound on where the information came from.
""")

#### (A.2.5) Visualise the hidden-state rows for a phone span

Take the centre frames of `/s/` above and plot `hidden_states[layer][0, t0:t1, :]` directly
as a heatmap.

In [ ]:
fc = frames_by_center(tr.start, tr.end, T)
t0, t1 = fc[0], fc[-1] + 1
print(f"phone {tr.label!r} -> hidden_states[L][0, {t0}:{t1}, :]  = {t1-t0} frames x {cfg.hidden_size} dims")

show = [0, 1, 6, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.0 * len(show), 3.6))
for ax, li in zip(axes, show):
    blk = hs[li][0, t0:t1].numpy()
    v   = np.abs(blk).max()
    im  = ax.imshow(blk, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                    interpolation="nearest")
    ax.set_title(f"hidden_states[{li}]\n{tr.label} rows {t0}:{t1}", fontsize=10)
    ax.set_xlabel("hidden dim (768)")
    ax.set_yticks(range(t1 - t0)); ax.set_yticklabels(range(t0, t1), fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel("encoder frame")
    plt.colorbar(im, ax=ax, fraction=0.035)
fig.suptitle(f"wav2vec 2.0 base - hidden-state rows for /{tr.label}/", y=1.04)
fig.tight_layout(); plt.show()

# whole utterance, to confirm the alignment lands where it should
fig, axes = plt.subplots(2, 1, figsize=(16, 6.4),
                         gridspec_kw={"height_ratios": [1, 1.5]})
plot_wave_with_phones(x, phones, ax=axes[0], title=f"wav2vec 2.0 base - waveform + PHN")
axes[0].axvspan(tr.t_start, tr.t_end, color="k", alpha=0.18, zorder=1)

H = hs[6][0].numpy()
v = np.percentile(np.abs(H), 99)
axes[1].imshow(H.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[0, T * HOP / SR, 0, H.shape[1]], interpolation="nearest")
for r in phones.itertuples():
    axes[1].axvline(r.t_start, color="k", lw=0.5, alpha=0.45)
axes[1].axvspan(tr.t_start, tr.t_end, facecolor="none", edgecolor="lime", lw=2.2)
axes[1].set_xlim(0, len(x) / SR)
axes[1].set_title("hidden_states[6]  (transposed: dim x frame); black = PHN boundaries, "
                  "green box = target phone")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("hidden dim")
fig.tight_layout(); plt.show()

### A.3 Probe the frozen model's layers — 探索冻结模型的各层表示

#### (A.3.1) Freeze and `eval()`

`eval()` turns off dropout and SpecAugment (`apply_spec_augment` only fires while training);
`requires_grad = False` cuts the gradients. The two are **independent** — do both.

In [ ]:
MODEL_KEY, HF_ID = "wav2vec2-base", "facebook/wav2vec2-base"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")

fe, model, cfg = load_model(HF_ID)

model.eval()
for p in model.parameters():
    p.requires_grad = False

n_all   = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"wav2vec 2.0 base  ({HF_ID})")
print(f"  model.training                  = {model.training}   (False -> dropout / SpecAugment off)")
print(f"  total parameters                = {n_all/1e6:.1f} M")
print(f"  parameters with requires_grad   = {n_train}   -> fully frozen: {n_train == 0}")
print(f"  config.apply_spec_augment       = {getattr(cfg, 'apply_spec_augment', None)} "
      f"(only takes effect while training=True)")

#### (A.3.2) Forward pass with `output_hidden_states=True`

In [ ]:
with torch.no_grad():
    inputs = fe(x, sampling_rate=SR, return_tensors="pt")
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

hs = outputs.hidden_states
T  = outputs.last_hidden_state.shape[1]
print("outputs fields:", list(outputs.keys()))
print(f"hidden_states: {len(hs)} x {tuple(hs[0].shape)}   T={T} frames "
      f"({T*HOP/SR:.3f} s @ {HOP/SR*1000:.0f} ms/frame)")
print("gradients detached (requires_grad):", outputs.last_hidden_state.requires_grad)

ftab          = frame_label_table(T, phones)
frame_phones  = ftab.phone.values
frame_classes = ftab["class"].values
print("\nper-frame label distribution:")
display(ftab["class"].value_counts().to_frame("n_frames").T)

#### (A.3.3) Per-layer statistics

The scale of each layer. The final layer's norm often blows up noticeably — which bears
directly on *which* layer to take as downstream features, and whether to layer-norm first.

In [ ]:
st = layer_stats(hs)
display(st.style.format({"mean": "{:+.5f}", "std": "{:.4f}", "mean_row_norm": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(st.layer, st.mean_row_norm, "o-", color="#4C78A8")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("mean ||h_t||")
axes[0].set_title(f"wav2vec 2.0 base - mean L2 norm per frame vector"); axes[0].grid(alpha=0.3)
axes[1].plot(st.layer, st["std"], "o-", color="#E45756")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("std")
axes[1].set_title("activation standard deviation"); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

#### (A.3.4) How similar are the layers: cosine and linear CKA

CKA is invariant to rotation and scaling, which makes it the standard choice for comparing
representations; the cosine matrix shows more directly which layers barely change anything.

In [ ]:
Mcos = layer_cosine(hs)
Mcka = layer_cka(hs)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, M, name in zip(axes, [Mcos, Mcka], ["per-frame cosine (mean-centred)", "linear CKA"]):
    im = ax.imshow(M, cmap="viridis", vmin=np.min([Mcos.min(), 0]), vmax=1.0)
    ax.set_title(f"{name}"); ax.set_xlabel("layer"); ax.set_ylabel("layer")
    ax.set_xticks(range(len(hs))); ax.set_yticks(range(len(hs)))
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"wav2vec 2.0 base - representation similarity between layers", y=1.01)
fig.tight_layout(); plt.show()

adj = pd.DataFrame({"layer_pair": [f"L{i}->L{i+1}" for i in range(len(hs)-1)],
                    "cosine": np.diag(Mcos, 1).round(4),
                    "CKA":    np.diag(Mcka, 1).round(4)})
display(adj.T)
print("The closer an adjacent-layer CKA is to 1, the less that layer changed; the lowest "
      "pairs mark where the representation shifts most sharply.")

#### (A.3.5) Phonetic information layer by layer

Three angles, ordered from **most trustworthy** to **most in need of caution**:

1. **Adjacent-frame contrast** (trustworthy): cosine similarity of adjacent frames within a
   phone vs across a phone boundary. ~144 adjacent pairs is enough statistical power on a
   single utterance, and a positive gap independently confirms the alignment is correct.
2. **Self-similarity matrix** (qualitative): the `T×T` frame-by-frame cosine matrix, where
   phone segments show up as blocks along the diagonal.
3. **Broad-class separability** (treat with caution): silhouette plus a 5-fold linear probe.
   With **one utterance, 138 frames, 6 classes** this is very noisy — what is being
   demonstrated is the **method**. Do not read the layer ordering as reproducing published
   curves; that needs hundreds of utterances.

In [ ]:
bc = boundary_contrast(hs, frame_phones)
display(bc.style.format({"within_phone": "{:.4f}", "across_boundary": "{:.4f}",
                         "gap": "{:+.4f}", "boundary_AUC": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(bc.layer, bc.within_phone,    "o-", label="within phone",    color="#54A24B")
axes[0].plot(bc.layer, bc.across_boundary, "o-", label="across boundary", color="#E45756")
axes[0].fill_between(bc.layer, bc.across_boundary, bc.within_phone, alpha=0.15, color="#54A24B")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("adjacent-frame cosine")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title(f"wav2vec 2.0 base - adjacent-frame similarity")
axes[1].plot(bc.layer, bc.boundary_AUC, "o-", color="#4C78A8")
axes[1].axhline(0.5, ls="--", c="gray", lw=1)
axes[1].set_xlabel("layer"); axes[1].set_ylabel("AUC")
axes[1].set_title("phone-boundary detection AUC from adjacent-frame similarity")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"gap positive at every layer: {bool((bc.gap > 0).all())}  -> every layer retains phone "
      f"boundary information, and the alignment checks out")
print(f"largest gap: L{int(bc.gap.idxmax())} ({bc.gap.max():+.4f}), "
      f"smallest: L{int(bc.gap.idxmin())} ({bc.gap.min():+.4f})")

In [ ]:
# Self-similarity: T x T frame-by-frame cosine; phone segments appear as diagonal blocks
show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.3))
bnd = [r.t_start for r in phones.itertuples()]
for ax, li in zip(axes, show):
    v = Fn.normalize(hs[li][0].float(), dim=-1)
    S = (v @ v.T).numpy()
    ax.imshow(S, cmap="magma", origin="lower", vmin=np.percentile(S, 2), vmax=1.0,
              extent=[0, T * HOP / SR, 0, T * HOP / SR], interpolation="nearest")
    for b in bnd:
        ax.axvline(b, color="#66ccff", lw=0.4, alpha=0.55)
        ax.axhline(b, color="#66ccff", lw=0.4, alpha=0.55)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xlabel("time (s)")
    if ax is axes[0]:
        ax.set_ylabel("time (s)")
fig.suptitle(f"wav2vec 2.0 base - frame-by-frame self-similarity (blue lines = PHN boundaries)", y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
sep, keep, yk = phone_separability(hs, frame_classes, min_count=5)
display(sep.style.format({"silhouette_cos": "{:+.4f}", "probe_acc_5fold": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(sep.layer, sep.silhouette_cos, "o-", color="#B279A2")
axes[0].axhline(0, ls="--", c="gray", lw=1)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("silhouette (cosine)")
axes[0].set_title("cluster geometry of broad phone classes"); axes[0].grid(alpha=0.3)
axes[1].plot(sep.layer, sep.probe_acc_5fold, "o-", color="#F58518")
axes[1].axhline(pd.Series(yk).value_counts(normalize=True).max(), ls="--", c="gray", lw=1,
                label="majority-class baseline")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("5-fold accuracy")
axes[1].set_title("linear probe (broad phone class)"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle(f"wav2vec 2.0 base - per-layer phone separability "
             f"(ONE utterance, {keep.sum()} frames - very noisy)", y=1.05)
fig.tight_layout(); plt.show()

print(f"frames used: {keep.sum()} / {T}, classes: {sorted(set(yk))}")
print(f"best silhouette: L{int(sep.silhouette_cos.idxmax())}   "
      f"best probe: L{int(sep.probe_acc_5fold.idxmax())}")
print("""
WARNING: with a single utterance the layer ordering of these two curves is NOT reliable and
should not be compared against published results. To get a meaningful curve, wrap
forward_frozen over a few hundred utterances, accumulate per-frame features and labels until
you have ~1e5 frames, and split train/test by speaker.
""")

In [ ]:
# PCA: project each frame to 2-D, colour by broad phone class
from sklearn.decomposition import PCA

show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.0))
for ax, li in zip(axes, show):
    Z = PCA(n_components=2, random_state=0).fit_transform(hs[li][0].float().numpy())
    for c in sorted(set(frame_classes)):
        m = frame_classes == c
        ax.scatter(Z[m, 0], Z[m, 1], s=20, c=CLASS_COLOR[c], label=c,
                   edgecolors="none", alpha=0.85)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="best", frameon=False)
fig.suptitle(f"wav2vec 2.0 base - PCA of per-frame representations (colour = broad phone class)", y=1.03)
fig.tight_layout(); plt.show()

---

# B. WavLM base — `microsoft/wavlm-base`

`microsoft/wavlm-base` (LibriSpeech 960h). Architecturally a sibling of HuBERT base, but with
**gated relative position bias** in attention, and utterance mixing (denoising) added to the
pre-training objective.

* Its front end **differs from the others**: `do_normalize=**False**` and
  `return_attention_mask=**True**`. This is exactly why the checkpoint's own feature extractor
  must be used.
* The output carries `extract_features` (layer-normed conv features, 512-dim).

### B.1 Read one utterance — 看懂一条数据

> This task is model-independent (pure TIMIT parsing), so its output is the same in all five
> model sections. It is repeated so that the **WavLM base** section stands on its own.

#### (B.1.1) Read the SPHERE audio

Print the 1024-byte NIST header field by field first, then read the waveform and cross-check
it against a hand-rolled parse.

In [ ]:
MODEL_KEY, HF_ID = "wavlm-base", "microsoft/wavlm-base"
print(f"### WavLM base  (microsoft/wavlm-base) ###\n")

magic, header_bytes, F = read_sphere_header(UTT_STEM + ".WAV")
print(f"SPHERE magic = {magic}   header = {header_bytes} bytes")
for k, v in F.items():
    print(f"   {k:20s} {v}")

x, sr, how = read_sphere(UTT_STEM + ".WAV")
print(f"\nread via: {how}")
print(f"waveform: shape={x.shape}  dtype={x.dtype}  sr={sr}  "
      f"duration={len(x)/sr:.4f} s  range=[{x.min():.4f}, {x.max():.4f}]")

# Cross-check: manual header parse + raw PCM should match libsndfile exactly
raw = np.fromfile(UTT_STEM + ".WAV", dtype="<i2",
                  offset=header_bytes, count=F["sample_count"]).astype(np.float32) / 32768.0
print(f"manual parse == soundfile ? {np.allclose(raw, x)}   (max |diff| = {np.abs(raw-x).max():.2e})")
assert sr == 16000 and x.ndim == 1

#### (B.1.2) Read `.TXT` / `.WRD` / `.PHN`

All three are **sample indices over half-open intervals `[start, end)`**.

In [ ]:
txt_s, txt_e, transcript = read_timit_txt(UTT_STEM + ".TXT")
words  = read_timit_seg(UTT_STEM + ".WRD")
phones = read_timit_seg(UTT_STEM + ".PHN")

print(f".TXT  [{txt_s}, {txt_e})  {transcript!r}")
print(f".WRD  {len(words)} words      .PHN  {len(phones)} phones\n")

display(words)
display(phones)

print("phone classes present:", sorted(set(phones.label.map(phone_class))))

# --- three real TIMIT gotchas visible in this very utterance ---
ov = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i].end - words.iloc[i+1].start)
      for i in range(len(words)-1) if words.iloc[i].end > words.iloc[i+1].start]
gp = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i+1].start - words.iloc[i].end)
      for i in range(len(words)-1) if words.iloc[i].end < words.iloc[i+1].start]
print(f"\n[gotcha 1] word spans that OVERLAP (overlap in samples): {ov}")
print(f"[gotcha 2] GAPS between words (gap in samples): {gp}")
print(f"[gotcha 3] PHN covers up to {phones.end.iloc[-1]}, but the waveform has {len(x)} "
      f"samples -> the last {len(x)-phones.end.iloc[-1]} samples are unlabelled")
print("\nBy contrast PHN itself is GAPLESS (every end == the next start):",
      bool((phones.end.values[:-1] == phones.start.values[1:]).all()))

#### (B.1.3) Play the whole utterance

In [ ]:
print(transcript)
display(Audio(x, rate=SR))

#### (B.1.4) Plot the waveform with phone boundaries and labels overlaid

Phone tier on top (colour = broad phone class), word tier in blue below. The second panel
zooms into `she had your`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8.5))
plot_wave_with_phones(x, phones, words, ax=axes[0],
                      title=f"WavLM base | {os.path.basename(UTT_STEM)} — {transcript}")
legend_classes(axes[0])
plot_wave_with_phones(x, phones, words, ax=axes[1], t0=0.15, t1=1.10,
                      title="zoom: 0.15 - 1.10 s  (she had your)")
fig.tight_layout()
plt.show()

#### (B.1.5) Crop a phone and listen to just that segment

Because `.PHN`'s `[start, end)` *is* a pair of array indices, cropping is literally
`x[start:end]`. Below: each phone on its own first, then the same phones with **20 ms of
context** added — many phones are near-unrecognisable in isolation, closures and stops
especially.

In [ ]:
def crop_phone(i, pad_ms=0.0):
    """Crop audio by PHN row index; pad_ms adds that much context on each side."""
    r   = phones.loc[i]
    pad = int(pad_ms / 1000 * SR)
    a, b = max(0, r.start - pad), min(len(x), r.end + pad)
    return x[a:b], r


def play_phone(i, pad_ms=0.0):
    seg, r = crop_phone(i, pad_ms)
    print(f"[{i:2d}] {r.label:4s} ({phone_class(r.label):9s})  "
          f"[{r.start}, {r.end})  {r.dur_ms:6.1f} ms  "
          f"{'+' + str(int(pad_ms)) + 'ms ctx' if pad_ms else ''}")
    display(Audio(seg, rate=SR))


# a representative spread: fricative / vowel / stop closure / stop release
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]))

print("\n---- same phones, with 20 ms of context on each side ----")
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]), pad_ms=20)

print("\n---- cropping by word works the same way ----")
for wi in [6, 7]:
    w = words.loc[wi]
    print(f"word {w.label!r}  [{w.start}, {w.end})  {w.dur_ms:.0f} ms")
    display(Audio(x[w.start:w.end], rate=SR))

With `ipywidgets` installed (`pip install ipywidgets`) the next cell gives a **dropdown**
that plays the selected phone; without it, it falls back to printing a table of choices.

In [ ]:
try:
    import ipywidgets as W

    opts = [(f"{i:2d}  {r.label:5s} {r.dur_ms:6.1f} ms  [{r.start},{r.end})", i)
            for i, r in phones.iterrows()]
    dd  = W.Dropdown(options=opts, description="phone:", layout=W.Layout(width="420px"))
    pad = W.IntSlider(value=0, min=0, max=100, step=10, description="ctx (ms):")
    outw = W.Output()

    def _on(_=None):
        with outw:
            outw.clear_output()
            play_phone(dd.value, pad_ms=pad.value)

    dd.observe(_on, names="value"); pad.observe(_on, names="value")
    display(W.VBox([W.HBox([dd, pad]), outw])); _on()
except ImportError:
    print("ipywidgets not installed -- pick manually with play_phone(i). Available phones:\n")
    print(phones.assign(cls=phones.label.map(phone_class))
                [["label", "cls", "start", "end", "dur_ms"]].to_string())
    print("\ne.g.  play_phone(13, pad_ms=20)")

### B.2 Sample-to-model-frame alignment — 完成对齐

Goal: translate `.PHN`'s `[start_sample, end_sample)` into exact encoder frame indices for
**WavLM base**.

#### (B.2.1) Feed in one 16 kHz waveform

Preprocess with the checkpoint's **own** `AutoFeatureExtractor` — the `do_normalize` and
`return_attention_mask` policies differ per checkpoint, and hand-rolled normalisation fails
silently.

In [ ]:
MODEL_KEY, HF_ID = "wavlm-base", "microsoft/wavlm-base"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")
words    = read_timit_seg(UTT_STEM + ".WRD")
assert sr == 16000, f"model expects 16 kHz, got {sr}"

display(describe_frontend(HF_ID))

fe, model, cfg = load_model(HF_ID)
inputs = fe(x, sampling_rate=SR, return_tensors="pt")
print("feature extractor output:", {k: tuple(v.shape) for k, v in inputs.items()})
print(f"raw waveform      mean={x.mean():+.5f}  std={x.std():.5f}")
iv = inputs["input_values"][0].numpy()
print(f"fed to the model  mean={iv.mean():+.5f}  std={iv.std():.5f}   "
      f"(do_normalize={fe.do_normalize})")

#### (B.2.2) Print the conv output and every hidden-state shape

`model.feature_extractor` *is* the 7-layer conv stack; it returns `(B, 512, T)` — note that
it is **channel-first**. `output_hidden_states=True` yields **13** tensors:
`hidden_states[0]` is the input to the Transformer, `hidden_states[i]` is the output of
Transformer layer `i`, and `hidden_states[12] is last_hidden_state`.

In [ ]:
R  = forward_frozen(HF_ID, x)
hs = R["hidden_states"]
T  = R["T"]

print(f"waveform                       {tuple(R['inputs']['input_values'].shape)}   ({len(x)} samples)")
print(f"conv feature_extractor output  {tuple(R['conv'].shape)}   (B, C=512, T) channel-first")
print(f"conv transposed                {tuple(R['conv'].transpose(1,2).shape)}   (B, T, C)")
if getattr(R["out"], "extract_features", None) is not None:
    print(f"outputs.extract_features       {tuple(R['out'].extract_features.shape)}   (layer-normed conv features)")
print(f"last_hidden_state              {tuple(R['out'].last_hidden_state.shape)}")
print(f"\nhidden_states: {len(hs)} tensors (= 1 + num_hidden_layers = 1 + {cfg.num_hidden_layers})")
for i, h in enumerate(hs):
    tag = "after conv projection / before Transformer" if i == 0 else f"output of Transformer layer {i}"
    star = "   <- last_hidden_state" if i == len(hs) - 1 else ""
    print(f"   hidden_states[{i:2d}]  {tuple(h.shape)}   {tag}{star}")
print("\nhidden_states[-1] is last_hidden_state ?",
      torch.equal(hs[-1], R["out"].last_hidden_state))

# length: hand-derived formula == HF's internal helper == the actual output
n = len(x)
print(f"\nlength check  n={n} samples")
print(f"   hand-derived conv_out_len(n)               = {conv_out_len(n)}")
print(f"   HF _get_feat_extract_output_lengths(n)     = {int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])}")
print(f"   actual T                                   = {T}")
assert conv_out_len(n) == T == int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])
print(f"\nLast frame's receptive field ends at {HOP*(T-1)+WIN}; the waveform is {n} samples "
      f"-> the final {n-(HOP*(T-1)+WIN)} samples are dropped, since the convs do not pad")

#### (B.2.3) Map PHN's `[start_sample, end_sample)` onto encoder frames

Two mappings, for two different purposes:

* **overlap** — the receptive field intersects the phone. Permissive; frames get shared
  between phones. Use it to ask *"which frames ever saw this phone?"*
* **centre** — the receptive-field **centre** falls inside the phone. Strict, and it forms a
  **partition** of the frames (each frame belongs to exactly one phone). This is the one you
  must use to build per-frame classification labels.

In [ ]:
rows = []
for r in phones.itertuples():
    fo = frames_overlapping(r.start, r.end, T)
    fc = frames_by_center(r.start, r.end, T)
    rows.append((r.Index, r.label, phone_class(r.label), r.start, r.end, round(r.dur_ms, 1),
                 f"[{fo[0]},{fo[-1]}]" if fo else "-", len(fo),
                 f"[{fc[0]},{fc[-1]}]" if fc else "-", len(fc)))
align = pd.DataFrame(rows, columns=["i", "phone", "class", "start", "end", "dur_ms",
                                    "overlap_frames", "n_overlap",
                                    "center_frames", "n_center"])
display(align)

print(f"centre-mapped frames sum to {align.n_center.sum()};  T = {T}   "
      f"-> exactly a partition: {align.n_center.sum() == T}")
print(f"overlap-mapped frames sum to {align.n_overlap.sum()} (> T, boundary frames double-counted)")

lost = align[align.n_center == 0]
print(f"\nPhones that receive NO centre frame at all "
      f"(anything shorter than the 20 ms hop can vanish entirely):")
display(lost[["i", "phone", "class", "start", "end", "dur_ms", "overlap_frames"]])
print("This is the intrinsic cost of a 20 ms frame shift: in per-frame phone labels, "
      "these phones simply disappear.")

#### (B.2.4) Check whether a frame's centre really falls inside the target phone

Taking the `/s/` in `suit`, walk every overlap frame and print its receptive field, its
centre, whether that centre is inside the phone, and how much of the receptive field the
phone actually covers.

In [ ]:
TARGET = "s"                                    # try any other phone here
ti = int(align[align.phone == TARGET].i.iloc[0])
tr = phones.loc[ti]
print(f"target phone: [{ti}] {tr.label}  [{tr.start}, {tr.end})  {tr.dur_ms:.1f} ms "
      f"= [{tr.t_start:.4f}, {tr.t_end:.4f}] s\n")

chk = []
for t in frames_overlapping(tr.start, tr.end, T):
    a, b = frame_span(t)
    c    = frame_center(t)
    inside = bool(tr.start <= c < tr.end)
    ov   = max(0, min(b, tr.end) - max(a, tr.start))
    chk.append((t, a, b, c, round(c / SR, 4), inside, ov, round(100 * ov / WIN, 1)))
chk = pd.DataFrame(chk, columns=["frame", "rf_start", "rf_end", "center", "center_s",
                                 "center_inside_phone", "overlap_samples", "overlap_%"])
display(chk)
print("Look at the first and last rows: their receptive fields do intersect /s/, but their "
      "centres already lie outside it, so the centre mapping excludes them. That is "
      "precisely where the two mappings part ways.")

# the global per-frame label table
ftab = frame_label_table(T, phones)
display(ftab.head(10))
print("...")
display(ftab.tail(5))
assert (ftab.phone != "<none>").all(), "some frame centre falls outside PHN coverage"

**Empirical check.** The `[320t, 320t+400)` span above came out of pure convolution
arithmetic. Is it actually true? Replace the normalisation layers in the conv stack with
`Identity`, perturb **a single sample**, and see which frames change.

In [ ]:
probe_sample = 2440                       # an arbitrary sample
pred = [t for t in range(T) if frame_span(t)[0] <= probe_sample < frame_span(t)[1]]

# (a) normalisation removed -> pure conv arithmetic, should match the formula exactly
m2, n_repl = copy.deepcopy(model), 0
for mod in m2.feature_extractor.modules():
    for cn, ch in list(mod.named_children()):
        if isinstance(ch, (nn.GroupNorm, nn.LayerNorm, nn.BatchNorm1d)):
            setattr(mod, cn, nn.Identity()); n_repl += 1

xb = torch.randn(1, len(x)) * 0.05
xp = xb.clone(); xp[0, probe_sample] += 1.0
with torch.no_grad():
    d_pure = (m2.feature_extractor(xp) - m2.feature_extractor(xb)).abs().sum(1)[0]
got = torch.nonzero(d_pure > 1e-5).flatten().tolist()
print(f"replaced {n_repl} normalisation layer(s) in the conv stack")
print(f"perturbing sample {probe_sample} -> frames affected {got}   formula predicts {pred}   match: {got == pred}")

# (b) normalisation kept -> measure how much global information it leaks
with torch.no_grad():
    d_real = (model.feature_extractor(xp) - model.feature_extractor(xb)).abs().sum(1)[0]
mask = torch.ones(T, dtype=bool); mask[pred] = False
inside, outside = d_real[pred].sum().item(), d_real[mask].sum().item()
print(f"\nwith normalisation kept: change inside the receptive field {inside:.4f} / "
      f"outside {outside:.4f} = {inside/max(outside,1e-12):.1f}x")
print(f"normalisation in the conv stack: {[type(m).__name__ for m in model.feature_extractor.modules() if isinstance(m,(nn.GroupNorm,nn.LayerNorm))]}")
print("""
Takeaways:
  * Under pure convolution arithmetic the receptive field [320t, 320t+400) is EXACT --
    it matches for all five models.
  * But GroupNorm computes its statistics along the TIME AXIS, so every frame ends up
    dependent on the whole utterance. Note how small the ratio above is: for the four
    GroupNorm checkpoints it lands around 0.5x-1.5x, i.e. the normalisation pathway carries
    as much of the perturbation as the local receptive field does. data2vec instead uses
    per-frame LayerNorm (7 of them) and does not leak at all -- its ratio is ~1e13.
  * More important still: all of this holds only for the **conv output / hidden_states[0]**.
    Past the first self-attention block, every frame of hidden_states[i>=1] can in principle
    see the entire utterance -- the frame-to-phone alignment is a **positional**
    correspondence, not a bound on where the information came from.
""")

#### (B.2.5) Visualise the hidden-state rows for a phone span

Take the centre frames of `/s/` above and plot `hidden_states[layer][0, t0:t1, :]` directly
as a heatmap.

In [ ]:
fc = frames_by_center(tr.start, tr.end, T)
t0, t1 = fc[0], fc[-1] + 1
print(f"phone {tr.label!r} -> hidden_states[L][0, {t0}:{t1}, :]  = {t1-t0} frames x {cfg.hidden_size} dims")

show = [0, 1, 6, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.0 * len(show), 3.6))
for ax, li in zip(axes, show):
    blk = hs[li][0, t0:t1].numpy()
    v   = np.abs(blk).max()
    im  = ax.imshow(blk, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                    interpolation="nearest")
    ax.set_title(f"hidden_states[{li}]\n{tr.label} rows {t0}:{t1}", fontsize=10)
    ax.set_xlabel("hidden dim (768)")
    ax.set_yticks(range(t1 - t0)); ax.set_yticklabels(range(t0, t1), fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel("encoder frame")
    plt.colorbar(im, ax=ax, fraction=0.035)
fig.suptitle(f"WavLM base - hidden-state rows for /{tr.label}/", y=1.04)
fig.tight_layout(); plt.show()

# whole utterance, to confirm the alignment lands where it should
fig, axes = plt.subplots(2, 1, figsize=(16, 6.4),
                         gridspec_kw={"height_ratios": [1, 1.5]})
plot_wave_with_phones(x, phones, ax=axes[0], title=f"WavLM base - waveform + PHN")
axes[0].axvspan(tr.t_start, tr.t_end, color="k", alpha=0.18, zorder=1)

H = hs[6][0].numpy()
v = np.percentile(np.abs(H), 99)
axes[1].imshow(H.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[0, T * HOP / SR, 0, H.shape[1]], interpolation="nearest")
for r in phones.itertuples():
    axes[1].axvline(r.t_start, color="k", lw=0.5, alpha=0.45)
axes[1].axvspan(tr.t_start, tr.t_end, facecolor="none", edgecolor="lime", lw=2.2)
axes[1].set_xlim(0, len(x) / SR)
axes[1].set_title("hidden_states[6]  (transposed: dim x frame); black = PHN boundaries, "
                  "green box = target phone")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("hidden dim")
fig.tight_layout(); plt.show()

### B.3 Probe the frozen model's layers — 探索冻结模型的各层表示

#### (B.3.1) Freeze and `eval()`

`eval()` turns off dropout and SpecAugment (`apply_spec_augment` only fires while training);
`requires_grad = False` cuts the gradients. The two are **independent** — do both.

In [ ]:
MODEL_KEY, HF_ID = "wavlm-base", "microsoft/wavlm-base"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")

fe, model, cfg = load_model(HF_ID)

model.eval()
for p in model.parameters():
    p.requires_grad = False

n_all   = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WavLM base  ({HF_ID})")
print(f"  model.training                  = {model.training}   (False -> dropout / SpecAugment off)")
print(f"  total parameters                = {n_all/1e6:.1f} M")
print(f"  parameters with requires_grad   = {n_train}   -> fully frozen: {n_train == 0}")
print(f"  config.apply_spec_augment       = {getattr(cfg, 'apply_spec_augment', None)} "
      f"(only takes effect while training=True)")

#### (B.3.2) Forward pass with `output_hidden_states=True`

In [ ]:
with torch.no_grad():
    inputs = fe(x, sampling_rate=SR, return_tensors="pt")
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

hs = outputs.hidden_states
T  = outputs.last_hidden_state.shape[1]
print("outputs fields:", list(outputs.keys()))
print(f"hidden_states: {len(hs)} x {tuple(hs[0].shape)}   T={T} frames "
      f"({T*HOP/SR:.3f} s @ {HOP/SR*1000:.0f} ms/frame)")
print("gradients detached (requires_grad):", outputs.last_hidden_state.requires_grad)

ftab          = frame_label_table(T, phones)
frame_phones  = ftab.phone.values
frame_classes = ftab["class"].values
print("\nper-frame label distribution:")
display(ftab["class"].value_counts().to_frame("n_frames").T)

#### (B.3.3) Per-layer statistics

The scale of each layer. The final layer's norm often blows up noticeably — which bears
directly on *which* layer to take as downstream features, and whether to layer-norm first.

In [ ]:
st = layer_stats(hs)
display(st.style.format({"mean": "{:+.5f}", "std": "{:.4f}", "mean_row_norm": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(st.layer, st.mean_row_norm, "o-", color="#4C78A8")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("mean ||h_t||")
axes[0].set_title(f"WavLM base - mean L2 norm per frame vector"); axes[0].grid(alpha=0.3)
axes[1].plot(st.layer, st["std"], "o-", color="#E45756")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("std")
axes[1].set_title("activation standard deviation"); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

#### (B.3.4) How similar are the layers: cosine and linear CKA

CKA is invariant to rotation and scaling, which makes it the standard choice for comparing
representations; the cosine matrix shows more directly which layers barely change anything.

In [ ]:
Mcos = layer_cosine(hs)
Mcka = layer_cka(hs)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, M, name in zip(axes, [Mcos, Mcka], ["per-frame cosine (mean-centred)", "linear CKA"]):
    im = ax.imshow(M, cmap="viridis", vmin=np.min([Mcos.min(), 0]), vmax=1.0)
    ax.set_title(f"{name}"); ax.set_xlabel("layer"); ax.set_ylabel("layer")
    ax.set_xticks(range(len(hs))); ax.set_yticks(range(len(hs)))
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"WavLM base - representation similarity between layers", y=1.01)
fig.tight_layout(); plt.show()

adj = pd.DataFrame({"layer_pair": [f"L{i}->L{i+1}" for i in range(len(hs)-1)],
                    "cosine": np.diag(Mcos, 1).round(4),
                    "CKA":    np.diag(Mcka, 1).round(4)})
display(adj.T)
print("The closer an adjacent-layer CKA is to 1, the less that layer changed; the lowest "
      "pairs mark where the representation shifts most sharply.")

#### (B.3.5) Phonetic information layer by layer

Three angles, ordered from **most trustworthy** to **most in need of caution**:

1. **Adjacent-frame contrast** (trustworthy): cosine similarity of adjacent frames within a
   phone vs across a phone boundary. ~144 adjacent pairs is enough statistical power on a
   single utterance, and a positive gap independently confirms the alignment is correct.
2. **Self-similarity matrix** (qualitative): the `T×T` frame-by-frame cosine matrix, where
   phone segments show up as blocks along the diagonal.
3. **Broad-class separability** (treat with caution): silhouette plus a 5-fold linear probe.
   With **one utterance, 138 frames, 6 classes** this is very noisy — what is being
   demonstrated is the **method**. Do not read the layer ordering as reproducing published
   curves; that needs hundreds of utterances.

In [ ]:
bc = boundary_contrast(hs, frame_phones)
display(bc.style.format({"within_phone": "{:.4f}", "across_boundary": "{:.4f}",
                         "gap": "{:+.4f}", "boundary_AUC": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(bc.layer, bc.within_phone,    "o-", label="within phone",    color="#54A24B")
axes[0].plot(bc.layer, bc.across_boundary, "o-", label="across boundary", color="#E45756")
axes[0].fill_between(bc.layer, bc.across_boundary, bc.within_phone, alpha=0.15, color="#54A24B")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("adjacent-frame cosine")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title(f"WavLM base - adjacent-frame similarity")
axes[1].plot(bc.layer, bc.boundary_AUC, "o-", color="#4C78A8")
axes[1].axhline(0.5, ls="--", c="gray", lw=1)
axes[1].set_xlabel("layer"); axes[1].set_ylabel("AUC")
axes[1].set_title("phone-boundary detection AUC from adjacent-frame similarity")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"gap positive at every layer: {bool((bc.gap > 0).all())}  -> every layer retains phone "
      f"boundary information, and the alignment checks out")
print(f"largest gap: L{int(bc.gap.idxmax())} ({bc.gap.max():+.4f}), "
      f"smallest: L{int(bc.gap.idxmin())} ({bc.gap.min():+.4f})")

In [ ]:
# Self-similarity: T x T frame-by-frame cosine; phone segments appear as diagonal blocks
show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.3))
bnd = [r.t_start for r in phones.itertuples()]
for ax, li in zip(axes, show):
    v = Fn.normalize(hs[li][0].float(), dim=-1)
    S = (v @ v.T).numpy()
    ax.imshow(S, cmap="magma", origin="lower", vmin=np.percentile(S, 2), vmax=1.0,
              extent=[0, T * HOP / SR, 0, T * HOP / SR], interpolation="nearest")
    for b in bnd:
        ax.axvline(b, color="#66ccff", lw=0.4, alpha=0.55)
        ax.axhline(b, color="#66ccff", lw=0.4, alpha=0.55)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xlabel("time (s)")
    if ax is axes[0]:
        ax.set_ylabel("time (s)")
fig.suptitle(f"WavLM base - frame-by-frame self-similarity (blue lines = PHN boundaries)", y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
sep, keep, yk = phone_separability(hs, frame_classes, min_count=5)
display(sep.style.format({"silhouette_cos": "{:+.4f}", "probe_acc_5fold": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(sep.layer, sep.silhouette_cos, "o-", color="#B279A2")
axes[0].axhline(0, ls="--", c="gray", lw=1)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("silhouette (cosine)")
axes[0].set_title("cluster geometry of broad phone classes"); axes[0].grid(alpha=0.3)
axes[1].plot(sep.layer, sep.probe_acc_5fold, "o-", color="#F58518")
axes[1].axhline(pd.Series(yk).value_counts(normalize=True).max(), ls="--", c="gray", lw=1,
                label="majority-class baseline")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("5-fold accuracy")
axes[1].set_title("linear probe (broad phone class)"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle(f"WavLM base - per-layer phone separability "
             f"(ONE utterance, {keep.sum()} frames - very noisy)", y=1.05)
fig.tight_layout(); plt.show()

print(f"frames used: {keep.sum()} / {T}, classes: {sorted(set(yk))}")
print(f"best silhouette: L{int(sep.silhouette_cos.idxmax())}   "
      f"best probe: L{int(sep.probe_acc_5fold.idxmax())}")
print("""
WARNING: with a single utterance the layer ordering of these two curves is NOT reliable and
should not be compared against published results. To get a meaningful curve, wrap
forward_frozen over a few hundred utterances, accumulate per-frame features and labels until
you have ~1e5 frames, and split train/test by speaker.
""")

In [ ]:
# PCA: project each frame to 2-D, colour by broad phone class
from sklearn.decomposition import PCA

show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.0))
for ax, li in zip(axes, show):
    Z = PCA(n_components=2, random_state=0).fit_transform(hs[li][0].float().numpy())
    for c in sorted(set(frame_classes)):
        m = frame_classes == c
        ax.scatter(Z[m, 0], Z[m, 1], s=20, c=CLASS_COLOR[c], label=c,
                   edgecolors="none", alpha=0.85)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="best", frameon=False)
fig.suptitle(f"WavLM base - PCA of per-frame representations (colour = broad phone class)", y=1.03)
fig.tight_layout(); plt.show()

---

# C. WavLM base+ — `microsoft/wavlm-base-plus`

`microsoft/wavlm-base-plus`: **structurally identical** to WavLM base; the only difference is
the pre-training data — 94k hours (Libri-Light 60k + VoxPopuli 24k + GigaSpeech 10k) against
base's 960 hours.

Sections B and C are therefore a clean **same-architecture, different-data-scale** pair:
shapes, alignment and layer count are all identical, so any difference can only show up in the
representation statistics of task 3.

### C.1 Read one utterance — 看懂一条数据

> This task is model-independent (pure TIMIT parsing), so its output is the same in all five
> model sections. It is repeated so that the **WavLM base+** section stands on its own.

#### (C.1.1) Read the SPHERE audio

Print the 1024-byte NIST header field by field first, then read the waveform and cross-check
it against a hand-rolled parse.

In [ ]:
MODEL_KEY, HF_ID = "wavlm-base-plus", "microsoft/wavlm-base-plus"
print(f"### WavLM base+  (microsoft/wavlm-base-plus) ###\n")

magic, header_bytes, F = read_sphere_header(UTT_STEM + ".WAV")
print(f"SPHERE magic = {magic}   header = {header_bytes} bytes")
for k, v in F.items():
    print(f"   {k:20s} {v}")

x, sr, how = read_sphere(UTT_STEM + ".WAV")
print(f"\nread via: {how}")
print(f"waveform: shape={x.shape}  dtype={x.dtype}  sr={sr}  "
      f"duration={len(x)/sr:.4f} s  range=[{x.min():.4f}, {x.max():.4f}]")

# Cross-check: manual header parse + raw PCM should match libsndfile exactly
raw = np.fromfile(UTT_STEM + ".WAV", dtype="<i2",
                  offset=header_bytes, count=F["sample_count"]).astype(np.float32) / 32768.0
print(f"manual parse == soundfile ? {np.allclose(raw, x)}   (max |diff| = {np.abs(raw-x).max():.2e})")
assert sr == 16000 and x.ndim == 1

#### (C.1.2) Read `.TXT` / `.WRD` / `.PHN`

All three are **sample indices over half-open intervals `[start, end)`**.

In [ ]:
txt_s, txt_e, transcript = read_timit_txt(UTT_STEM + ".TXT")
words  = read_timit_seg(UTT_STEM + ".WRD")
phones = read_timit_seg(UTT_STEM + ".PHN")

print(f".TXT  [{txt_s}, {txt_e})  {transcript!r}")
print(f".WRD  {len(words)} words      .PHN  {len(phones)} phones\n")

display(words)
display(phones)

print("phone classes present:", sorted(set(phones.label.map(phone_class))))

# --- three real TIMIT gotchas visible in this very utterance ---
ov = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i].end - words.iloc[i+1].start)
      for i in range(len(words)-1) if words.iloc[i].end > words.iloc[i+1].start]
gp = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i+1].start - words.iloc[i].end)
      for i in range(len(words)-1) if words.iloc[i].end < words.iloc[i+1].start]
print(f"\n[gotcha 1] word spans that OVERLAP (overlap in samples): {ov}")
print(f"[gotcha 2] GAPS between words (gap in samples): {gp}")
print(f"[gotcha 3] PHN covers up to {phones.end.iloc[-1]}, but the waveform has {len(x)} "
      f"samples -> the last {len(x)-phones.end.iloc[-1]} samples are unlabelled")
print("\nBy contrast PHN itself is GAPLESS (every end == the next start):",
      bool((phones.end.values[:-1] == phones.start.values[1:]).all()))

#### (C.1.3) Play the whole utterance

In [ ]:
print(transcript)
display(Audio(x, rate=SR))

#### (C.1.4) Plot the waveform with phone boundaries and labels overlaid

Phone tier on top (colour = broad phone class), word tier in blue below. The second panel
zooms into `she had your`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8.5))
plot_wave_with_phones(x, phones, words, ax=axes[0],
                      title=f"WavLM base+ | {os.path.basename(UTT_STEM)} — {transcript}")
legend_classes(axes[0])
plot_wave_with_phones(x, phones, words, ax=axes[1], t0=0.15, t1=1.10,
                      title="zoom: 0.15 - 1.10 s  (she had your)")
fig.tight_layout()
plt.show()

#### (C.1.5) Crop a phone and listen to just that segment

Because `.PHN`'s `[start, end)` *is* a pair of array indices, cropping is literally
`x[start:end]`. Below: each phone on its own first, then the same phones with **20 ms of
context** added — many phones are near-unrecognisable in isolation, closures and stops
especially.

In [ ]:
def crop_phone(i, pad_ms=0.0):
    """Crop audio by PHN row index; pad_ms adds that much context on each side."""
    r   = phones.loc[i]
    pad = int(pad_ms / 1000 * SR)
    a, b = max(0, r.start - pad), min(len(x), r.end + pad)
    return x[a:b], r


def play_phone(i, pad_ms=0.0):
    seg, r = crop_phone(i, pad_ms)
    print(f"[{i:2d}] {r.label:4s} ({phone_class(r.label):9s})  "
          f"[{r.start}, {r.end})  {r.dur_ms:6.1f} ms  "
          f"{'+' + str(int(pad_ms)) + 'ms ctx' if pad_ms else ''}")
    display(Audio(seg, rate=SR))


# a representative spread: fricative / vowel / stop closure / stop release
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]))

print("\n---- same phones, with 20 ms of context on each side ----")
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]), pad_ms=20)

print("\n---- cropping by word works the same way ----")
for wi in [6, 7]:
    w = words.loc[wi]
    print(f"word {w.label!r}  [{w.start}, {w.end})  {w.dur_ms:.0f} ms")
    display(Audio(x[w.start:w.end], rate=SR))

With `ipywidgets` installed (`pip install ipywidgets`) the next cell gives a **dropdown**
that plays the selected phone; without it, it falls back to printing a table of choices.

In [ ]:
try:
    import ipywidgets as W

    opts = [(f"{i:2d}  {r.label:5s} {r.dur_ms:6.1f} ms  [{r.start},{r.end})", i)
            for i, r in phones.iterrows()]
    dd  = W.Dropdown(options=opts, description="phone:", layout=W.Layout(width="420px"))
    pad = W.IntSlider(value=0, min=0, max=100, step=10, description="ctx (ms):")
    outw = W.Output()

    def _on(_=None):
        with outw:
            outw.clear_output()
            play_phone(dd.value, pad_ms=pad.value)

    dd.observe(_on, names="value"); pad.observe(_on, names="value")
    display(W.VBox([W.HBox([dd, pad]), outw])); _on()
except ImportError:
    print("ipywidgets not installed -- pick manually with play_phone(i). Available phones:\n")
    print(phones.assign(cls=phones.label.map(phone_class))
                [["label", "cls", "start", "end", "dur_ms"]].to_string())
    print("\ne.g.  play_phone(13, pad_ms=20)")

### C.2 Sample-to-model-frame alignment — 完成对齐

Goal: translate `.PHN`'s `[start_sample, end_sample)` into exact encoder frame indices for
**WavLM base+**.

#### (C.2.1) Feed in one 16 kHz waveform

Preprocess with the checkpoint's **own** `AutoFeatureExtractor` — the `do_normalize` and
`return_attention_mask` policies differ per checkpoint, and hand-rolled normalisation fails
silently.

In [ ]:
MODEL_KEY, HF_ID = "wavlm-base-plus", "microsoft/wavlm-base-plus"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")
words    = read_timit_seg(UTT_STEM + ".WRD")
assert sr == 16000, f"model expects 16 kHz, got {sr}"

display(describe_frontend(HF_ID))

fe, model, cfg = load_model(HF_ID)
inputs = fe(x, sampling_rate=SR, return_tensors="pt")
print("feature extractor output:", {k: tuple(v.shape) for k, v in inputs.items()})
print(f"raw waveform      mean={x.mean():+.5f}  std={x.std():.5f}")
iv = inputs["input_values"][0].numpy()
print(f"fed to the model  mean={iv.mean():+.5f}  std={iv.std():.5f}   "
      f"(do_normalize={fe.do_normalize})")

#### (C.2.2) Print the conv output and every hidden-state shape

`model.feature_extractor` *is* the 7-layer conv stack; it returns `(B, 512, T)` — note that
it is **channel-first**. `output_hidden_states=True` yields **13** tensors:
`hidden_states[0]` is the input to the Transformer, `hidden_states[i]` is the output of
Transformer layer `i`, and `hidden_states[12] is last_hidden_state`.

In [ ]:
R  = forward_frozen(HF_ID, x)
hs = R["hidden_states"]
T  = R["T"]

print(f"waveform                       {tuple(R['inputs']['input_values'].shape)}   ({len(x)} samples)")
print(f"conv feature_extractor output  {tuple(R['conv'].shape)}   (B, C=512, T) channel-first")
print(f"conv transposed                {tuple(R['conv'].transpose(1,2).shape)}   (B, T, C)")
if getattr(R["out"], "extract_features", None) is not None:
    print(f"outputs.extract_features       {tuple(R['out'].extract_features.shape)}   (layer-normed conv features)")
print(f"last_hidden_state              {tuple(R['out'].last_hidden_state.shape)}")
print(f"\nhidden_states: {len(hs)} tensors (= 1 + num_hidden_layers = 1 + {cfg.num_hidden_layers})")
for i, h in enumerate(hs):
    tag = "after conv projection / before Transformer" if i == 0 else f"output of Transformer layer {i}"
    star = "   <- last_hidden_state" if i == len(hs) - 1 else ""
    print(f"   hidden_states[{i:2d}]  {tuple(h.shape)}   {tag}{star}")
print("\nhidden_states[-1] is last_hidden_state ?",
      torch.equal(hs[-1], R["out"].last_hidden_state))

# length: hand-derived formula == HF's internal helper == the actual output
n = len(x)
print(f"\nlength check  n={n} samples")
print(f"   hand-derived conv_out_len(n)               = {conv_out_len(n)}")
print(f"   HF _get_feat_extract_output_lengths(n)     = {int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])}")
print(f"   actual T                                   = {T}")
assert conv_out_len(n) == T == int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])
print(f"\nLast frame's receptive field ends at {HOP*(T-1)+WIN}; the waveform is {n} samples "
      f"-> the final {n-(HOP*(T-1)+WIN)} samples are dropped, since the convs do not pad")

#### (C.2.3) Map PHN's `[start_sample, end_sample)` onto encoder frames

Two mappings, for two different purposes:

* **overlap** — the receptive field intersects the phone. Permissive; frames get shared
  between phones. Use it to ask *"which frames ever saw this phone?"*
* **centre** — the receptive-field **centre** falls inside the phone. Strict, and it forms a
  **partition** of the frames (each frame belongs to exactly one phone). This is the one you
  must use to build per-frame classification labels.

In [ ]:
rows = []
for r in phones.itertuples():
    fo = frames_overlapping(r.start, r.end, T)
    fc = frames_by_center(r.start, r.end, T)
    rows.append((r.Index, r.label, phone_class(r.label), r.start, r.end, round(r.dur_ms, 1),
                 f"[{fo[0]},{fo[-1]}]" if fo else "-", len(fo),
                 f"[{fc[0]},{fc[-1]}]" if fc else "-", len(fc)))
align = pd.DataFrame(rows, columns=["i", "phone", "class", "start", "end", "dur_ms",
                                    "overlap_frames", "n_overlap",
                                    "center_frames", "n_center"])
display(align)

print(f"centre-mapped frames sum to {align.n_center.sum()};  T = {T}   "
      f"-> exactly a partition: {align.n_center.sum() == T}")
print(f"overlap-mapped frames sum to {align.n_overlap.sum()} (> T, boundary frames double-counted)")

lost = align[align.n_center == 0]
print(f"\nPhones that receive NO centre frame at all "
      f"(anything shorter than the 20 ms hop can vanish entirely):")
display(lost[["i", "phone", "class", "start", "end", "dur_ms", "overlap_frames"]])
print("This is the intrinsic cost of a 20 ms frame shift: in per-frame phone labels, "
      "these phones simply disappear.")

#### (C.2.4) Check whether a frame's centre really falls inside the target phone

Taking the `/s/` in `suit`, walk every overlap frame and print its receptive field, its
centre, whether that centre is inside the phone, and how much of the receptive field the
phone actually covers.

In [ ]:
TARGET = "s"                                    # try any other phone here
ti = int(align[align.phone == TARGET].i.iloc[0])
tr = phones.loc[ti]
print(f"target phone: [{ti}] {tr.label}  [{tr.start}, {tr.end})  {tr.dur_ms:.1f} ms "
      f"= [{tr.t_start:.4f}, {tr.t_end:.4f}] s\n")

chk = []
for t in frames_overlapping(tr.start, tr.end, T):
    a, b = frame_span(t)
    c    = frame_center(t)
    inside = bool(tr.start <= c < tr.end)
    ov   = max(0, min(b, tr.end) - max(a, tr.start))
    chk.append((t, a, b, c, round(c / SR, 4), inside, ov, round(100 * ov / WIN, 1)))
chk = pd.DataFrame(chk, columns=["frame", "rf_start", "rf_end", "center", "center_s",
                                 "center_inside_phone", "overlap_samples", "overlap_%"])
display(chk)
print("Look at the first and last rows: their receptive fields do intersect /s/, but their "
      "centres already lie outside it, so the centre mapping excludes them. That is "
      "precisely where the two mappings part ways.")

# the global per-frame label table
ftab = frame_label_table(T, phones)
display(ftab.head(10))
print("...")
display(ftab.tail(5))
assert (ftab.phone != "<none>").all(), "some frame centre falls outside PHN coverage"

**Empirical check.** The `[320t, 320t+400)` span above came out of pure convolution
arithmetic. Is it actually true? Replace the normalisation layers in the conv stack with
`Identity`, perturb **a single sample**, and see which frames change.

In [ ]:
probe_sample = 2440                       # an arbitrary sample
pred = [t for t in range(T) if frame_span(t)[0] <= probe_sample < frame_span(t)[1]]

# (a) normalisation removed -> pure conv arithmetic, should match the formula exactly
m2, n_repl = copy.deepcopy(model), 0
for mod in m2.feature_extractor.modules():
    for cn, ch in list(mod.named_children()):
        if isinstance(ch, (nn.GroupNorm, nn.LayerNorm, nn.BatchNorm1d)):
            setattr(mod, cn, nn.Identity()); n_repl += 1

xb = torch.randn(1, len(x)) * 0.05
xp = xb.clone(); xp[0, probe_sample] += 1.0
with torch.no_grad():
    d_pure = (m2.feature_extractor(xp) - m2.feature_extractor(xb)).abs().sum(1)[0]
got = torch.nonzero(d_pure > 1e-5).flatten().tolist()
print(f"replaced {n_repl} normalisation layer(s) in the conv stack")
print(f"perturbing sample {probe_sample} -> frames affected {got}   formula predicts {pred}   match: {got == pred}")

# (b) normalisation kept -> measure how much global information it leaks
with torch.no_grad():
    d_real = (model.feature_extractor(xp) - model.feature_extractor(xb)).abs().sum(1)[0]
mask = torch.ones(T, dtype=bool); mask[pred] = False
inside, outside = d_real[pred].sum().item(), d_real[mask].sum().item()
print(f"\nwith normalisation kept: change inside the receptive field {inside:.4f} / "
      f"outside {outside:.4f} = {inside/max(outside,1e-12):.1f}x")
print(f"normalisation in the conv stack: {[type(m).__name__ for m in model.feature_extractor.modules() if isinstance(m,(nn.GroupNorm,nn.LayerNorm))]}")
print("""
Takeaways:
  * Under pure convolution arithmetic the receptive field [320t, 320t+400) is EXACT --
    it matches for all five models.
  * But GroupNorm computes its statistics along the TIME AXIS, so every frame ends up
    dependent on the whole utterance. Note how small the ratio above is: for the four
    GroupNorm checkpoints it lands around 0.5x-1.5x, i.e. the normalisation pathway carries
    as much of the perturbation as the local receptive field does. data2vec instead uses
    per-frame LayerNorm (7 of them) and does not leak at all -- its ratio is ~1e13.
  * More important still: all of this holds only for the **conv output / hidden_states[0]**.
    Past the first self-attention block, every frame of hidden_states[i>=1] can in principle
    see the entire utterance -- the frame-to-phone alignment is a **positional**
    correspondence, not a bound on where the information came from.
""")

#### (C.2.5) Visualise the hidden-state rows for a phone span

Take the centre frames of `/s/` above and plot `hidden_states[layer][0, t0:t1, :]` directly
as a heatmap.

In [ ]:
fc = frames_by_center(tr.start, tr.end, T)
t0, t1 = fc[0], fc[-1] + 1
print(f"phone {tr.label!r} -> hidden_states[L][0, {t0}:{t1}, :]  = {t1-t0} frames x {cfg.hidden_size} dims")

show = [0, 1, 6, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.0 * len(show), 3.6))
for ax, li in zip(axes, show):
    blk = hs[li][0, t0:t1].numpy()
    v   = np.abs(blk).max()
    im  = ax.imshow(blk, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                    interpolation="nearest")
    ax.set_title(f"hidden_states[{li}]\n{tr.label} rows {t0}:{t1}", fontsize=10)
    ax.set_xlabel("hidden dim (768)")
    ax.set_yticks(range(t1 - t0)); ax.set_yticklabels(range(t0, t1), fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel("encoder frame")
    plt.colorbar(im, ax=ax, fraction=0.035)
fig.suptitle(f"WavLM base+ - hidden-state rows for /{tr.label}/", y=1.04)
fig.tight_layout(); plt.show()

# whole utterance, to confirm the alignment lands where it should
fig, axes = plt.subplots(2, 1, figsize=(16, 6.4),
                         gridspec_kw={"height_ratios": [1, 1.5]})
plot_wave_with_phones(x, phones, ax=axes[0], title=f"WavLM base+ - waveform + PHN")
axes[0].axvspan(tr.t_start, tr.t_end, color="k", alpha=0.18, zorder=1)

H = hs[6][0].numpy()
v = np.percentile(np.abs(H), 99)
axes[1].imshow(H.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[0, T * HOP / SR, 0, H.shape[1]], interpolation="nearest")
for r in phones.itertuples():
    axes[1].axvline(r.t_start, color="k", lw=0.5, alpha=0.45)
axes[1].axvspan(tr.t_start, tr.t_end, facecolor="none", edgecolor="lime", lw=2.2)
axes[1].set_xlim(0, len(x) / SR)
axes[1].set_title("hidden_states[6]  (transposed: dim x frame); black = PHN boundaries, "
                  "green box = target phone")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("hidden dim")
fig.tight_layout(); plt.show()

### C.3 Probe the frozen model's layers — 探索冻结模型的各层表示

#### (C.3.1) Freeze and `eval()`

`eval()` turns off dropout and SpecAugment (`apply_spec_augment` only fires while training);
`requires_grad = False` cuts the gradients. The two are **independent** — do both.

In [ ]:
MODEL_KEY, HF_ID = "wavlm-base-plus", "microsoft/wavlm-base-plus"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")

fe, model, cfg = load_model(HF_ID)

model.eval()
for p in model.parameters():
    p.requires_grad = False

n_all   = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"WavLM base+  ({HF_ID})")
print(f"  model.training                  = {model.training}   (False -> dropout / SpecAugment off)")
print(f"  total parameters                = {n_all/1e6:.1f} M")
print(f"  parameters with requires_grad   = {n_train}   -> fully frozen: {n_train == 0}")
print(f"  config.apply_spec_augment       = {getattr(cfg, 'apply_spec_augment', None)} "
      f"(only takes effect while training=True)")

#### (C.3.2) Forward pass with `output_hidden_states=True`

In [ ]:
with torch.no_grad():
    inputs = fe(x, sampling_rate=SR, return_tensors="pt")
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

hs = outputs.hidden_states
T  = outputs.last_hidden_state.shape[1]
print("outputs fields:", list(outputs.keys()))
print(f"hidden_states: {len(hs)} x {tuple(hs[0].shape)}   T={T} frames "
      f"({T*HOP/SR:.3f} s @ {HOP/SR*1000:.0f} ms/frame)")
print("gradients detached (requires_grad):", outputs.last_hidden_state.requires_grad)

ftab          = frame_label_table(T, phones)
frame_phones  = ftab.phone.values
frame_classes = ftab["class"].values
print("\nper-frame label distribution:")
display(ftab["class"].value_counts().to_frame("n_frames").T)

#### (C.3.3) Per-layer statistics

The scale of each layer. The final layer's norm often blows up noticeably — which bears
directly on *which* layer to take as downstream features, and whether to layer-norm first.

In [ ]:
st = layer_stats(hs)
display(st.style.format({"mean": "{:+.5f}", "std": "{:.4f}", "mean_row_norm": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(st.layer, st.mean_row_norm, "o-", color="#4C78A8")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("mean ||h_t||")
axes[0].set_title(f"WavLM base+ - mean L2 norm per frame vector"); axes[0].grid(alpha=0.3)
axes[1].plot(st.layer, st["std"], "o-", color="#E45756")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("std")
axes[1].set_title("activation standard deviation"); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

#### (C.3.4) How similar are the layers: cosine and linear CKA

CKA is invariant to rotation and scaling, which makes it the standard choice for comparing
representations; the cosine matrix shows more directly which layers barely change anything.

In [ ]:
Mcos = layer_cosine(hs)
Mcka = layer_cka(hs)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, M, name in zip(axes, [Mcos, Mcka], ["per-frame cosine (mean-centred)", "linear CKA"]):
    im = ax.imshow(M, cmap="viridis", vmin=np.min([Mcos.min(), 0]), vmax=1.0)
    ax.set_title(f"{name}"); ax.set_xlabel("layer"); ax.set_ylabel("layer")
    ax.set_xticks(range(len(hs))); ax.set_yticks(range(len(hs)))
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"WavLM base+ - representation similarity between layers", y=1.01)
fig.tight_layout(); plt.show()

adj = pd.DataFrame({"layer_pair": [f"L{i}->L{i+1}" for i in range(len(hs)-1)],
                    "cosine": np.diag(Mcos, 1).round(4),
                    "CKA":    np.diag(Mcka, 1).round(4)})
display(adj.T)
print("The closer an adjacent-layer CKA is to 1, the less that layer changed; the lowest "
      "pairs mark where the representation shifts most sharply.")

#### (C.3.5) Phonetic information layer by layer

Three angles, ordered from **most trustworthy** to **most in need of caution**:

1. **Adjacent-frame contrast** (trustworthy): cosine similarity of adjacent frames within a
   phone vs across a phone boundary. ~144 adjacent pairs is enough statistical power on a
   single utterance, and a positive gap independently confirms the alignment is correct.
2. **Self-similarity matrix** (qualitative): the `T×T` frame-by-frame cosine matrix, where
   phone segments show up as blocks along the diagonal.
3. **Broad-class separability** (treat with caution): silhouette plus a 5-fold linear probe.
   With **one utterance, 138 frames, 6 classes** this is very noisy — what is being
   demonstrated is the **method**. Do not read the layer ordering as reproducing published
   curves; that needs hundreds of utterances.

In [ ]:
bc = boundary_contrast(hs, frame_phones)
display(bc.style.format({"within_phone": "{:.4f}", "across_boundary": "{:.4f}",
                         "gap": "{:+.4f}", "boundary_AUC": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(bc.layer, bc.within_phone,    "o-", label="within phone",    color="#54A24B")
axes[0].plot(bc.layer, bc.across_boundary, "o-", label="across boundary", color="#E45756")
axes[0].fill_between(bc.layer, bc.across_boundary, bc.within_phone, alpha=0.15, color="#54A24B")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("adjacent-frame cosine")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title(f"WavLM base+ - adjacent-frame similarity")
axes[1].plot(bc.layer, bc.boundary_AUC, "o-", color="#4C78A8")
axes[1].axhline(0.5, ls="--", c="gray", lw=1)
axes[1].set_xlabel("layer"); axes[1].set_ylabel("AUC")
axes[1].set_title("phone-boundary detection AUC from adjacent-frame similarity")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"gap positive at every layer: {bool((bc.gap > 0).all())}  -> every layer retains phone "
      f"boundary information, and the alignment checks out")
print(f"largest gap: L{int(bc.gap.idxmax())} ({bc.gap.max():+.4f}), "
      f"smallest: L{int(bc.gap.idxmin())} ({bc.gap.min():+.4f})")

In [ ]:
# Self-similarity: T x T frame-by-frame cosine; phone segments appear as diagonal blocks
show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.3))
bnd = [r.t_start for r in phones.itertuples()]
for ax, li in zip(axes, show):
    v = Fn.normalize(hs[li][0].float(), dim=-1)
    S = (v @ v.T).numpy()
    ax.imshow(S, cmap="magma", origin="lower", vmin=np.percentile(S, 2), vmax=1.0,
              extent=[0, T * HOP / SR, 0, T * HOP / SR], interpolation="nearest")
    for b in bnd:
        ax.axvline(b, color="#66ccff", lw=0.4, alpha=0.55)
        ax.axhline(b, color="#66ccff", lw=0.4, alpha=0.55)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xlabel("time (s)")
    if ax is axes[0]:
        ax.set_ylabel("time (s)")
fig.suptitle(f"WavLM base+ - frame-by-frame self-similarity (blue lines = PHN boundaries)", y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
sep, keep, yk = phone_separability(hs, frame_classes, min_count=5)
display(sep.style.format({"silhouette_cos": "{:+.4f}", "probe_acc_5fold": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(sep.layer, sep.silhouette_cos, "o-", color="#B279A2")
axes[0].axhline(0, ls="--", c="gray", lw=1)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("silhouette (cosine)")
axes[0].set_title("cluster geometry of broad phone classes"); axes[0].grid(alpha=0.3)
axes[1].plot(sep.layer, sep.probe_acc_5fold, "o-", color="#F58518")
axes[1].axhline(pd.Series(yk).value_counts(normalize=True).max(), ls="--", c="gray", lw=1,
                label="majority-class baseline")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("5-fold accuracy")
axes[1].set_title("linear probe (broad phone class)"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle(f"WavLM base+ - per-layer phone separability "
             f"(ONE utterance, {keep.sum()} frames - very noisy)", y=1.05)
fig.tight_layout(); plt.show()

print(f"frames used: {keep.sum()} / {T}, classes: {sorted(set(yk))}")
print(f"best silhouette: L{int(sep.silhouette_cos.idxmax())}   "
      f"best probe: L{int(sep.probe_acc_5fold.idxmax())}")
print("""
WARNING: with a single utterance the layer ordering of these two curves is NOT reliable and
should not be compared against published results. To get a meaningful curve, wrap
forward_frozen over a few hundred utterances, accumulate per-frame features and labels until
you have ~1e5 frames, and split train/test by speaker.
""")

In [ ]:
# PCA: project each frame to 2-D, colour by broad phone class
from sklearn.decomposition import PCA

show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.0))
for ax, li in zip(axes, show):
    Z = PCA(n_components=2, random_state=0).fit_transform(hs[li][0].float().numpy())
    for c in sorted(set(frame_classes)):
        m = frame_classes == c
        ax.scatter(Z[m, 0], Z[m, 1], s=20, c=CLASS_COLOR[c], label=c,
                   edgecolors="none", alpha=0.85)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="best", frameon=False)
fig.suptitle(f"WavLM base+ - PCA of per-frame representations (colour = broad phone class)", y=1.03)
fig.tight_layout(); plt.show()

---

# D. HuBERT base — `facebook/hubert-base-ls960`

`facebook/hubert-base-ls960` (LibriSpeech 960h), masked prediction over k-means pseudo-labels.

* `AutoModel` gives a `HubertModel`; unlike wav2vec2 / WavLM / data2vec, its output has **no**
  `extract_features` field.
* Front end: `do_normalize=True`, `return_attention_mask=False`.
* The first conv layer is likewise a **GroupNorm**.

### D.1 Read one utterance — 看懂一条数据

> This task is model-independent (pure TIMIT parsing), so its output is the same in all five
> model sections. It is repeated so that the **HuBERT base** section stands on its own.

#### (D.1.1) Read the SPHERE audio

Print the 1024-byte NIST header field by field first, then read the waveform and cross-check
it against a hand-rolled parse.

In [ ]:
MODEL_KEY, HF_ID = "hubert-base", "facebook/hubert-base-ls960"
print(f"### HuBERT base  (facebook/hubert-base-ls960) ###\n")

magic, header_bytes, F = read_sphere_header(UTT_STEM + ".WAV")
print(f"SPHERE magic = {magic}   header = {header_bytes} bytes")
for k, v in F.items():
    print(f"   {k:20s} {v}")

x, sr, how = read_sphere(UTT_STEM + ".WAV")
print(f"\nread via: {how}")
print(f"waveform: shape={x.shape}  dtype={x.dtype}  sr={sr}  "
      f"duration={len(x)/sr:.4f} s  range=[{x.min():.4f}, {x.max():.4f}]")

# Cross-check: manual header parse + raw PCM should match libsndfile exactly
raw = np.fromfile(UTT_STEM + ".WAV", dtype="<i2",
                  offset=header_bytes, count=F["sample_count"]).astype(np.float32) / 32768.0
print(f"manual parse == soundfile ? {np.allclose(raw, x)}   (max |diff| = {np.abs(raw-x).max():.2e})")
assert sr == 16000 and x.ndim == 1

#### (D.1.2) Read `.TXT` / `.WRD` / `.PHN`

All three are **sample indices over half-open intervals `[start, end)`**.

In [ ]:
txt_s, txt_e, transcript = read_timit_txt(UTT_STEM + ".TXT")
words  = read_timit_seg(UTT_STEM + ".WRD")
phones = read_timit_seg(UTT_STEM + ".PHN")

print(f".TXT  [{txt_s}, {txt_e})  {transcript!r}")
print(f".WRD  {len(words)} words      .PHN  {len(phones)} phones\n")

display(words)
display(phones)

print("phone classes present:", sorted(set(phones.label.map(phone_class))))

# --- three real TIMIT gotchas visible in this very utterance ---
ov = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i].end - words.iloc[i+1].start)
      for i in range(len(words)-1) if words.iloc[i].end > words.iloc[i+1].start]
gp = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i+1].start - words.iloc[i].end)
      for i in range(len(words)-1) if words.iloc[i].end < words.iloc[i+1].start]
print(f"\n[gotcha 1] word spans that OVERLAP (overlap in samples): {ov}")
print(f"[gotcha 2] GAPS between words (gap in samples): {gp}")
print(f"[gotcha 3] PHN covers up to {phones.end.iloc[-1]}, but the waveform has {len(x)} "
      f"samples -> the last {len(x)-phones.end.iloc[-1]} samples are unlabelled")
print("\nBy contrast PHN itself is GAPLESS (every end == the next start):",
      bool((phones.end.values[:-1] == phones.start.values[1:]).all()))

#### (D.1.3) Play the whole utterance

In [ ]:
print(transcript)
display(Audio(x, rate=SR))

#### (D.1.4) Plot the waveform with phone boundaries and labels overlaid

Phone tier on top (colour = broad phone class), word tier in blue below. The second panel
zooms into `she had your`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8.5))
plot_wave_with_phones(x, phones, words, ax=axes[0],
                      title=f"HuBERT base | {os.path.basename(UTT_STEM)} — {transcript}")
legend_classes(axes[0])
plot_wave_with_phones(x, phones, words, ax=axes[1], t0=0.15, t1=1.10,
                      title="zoom: 0.15 - 1.10 s  (she had your)")
fig.tight_layout()
plt.show()

#### (D.1.5) Crop a phone and listen to just that segment

Because `.PHN`'s `[start, end)` *is* a pair of array indices, cropping is literally
`x[start:end]`. Below: each phone on its own first, then the same phones with **20 ms of
context** added — many phones are near-unrecognisable in isolation, closures and stops
especially.

In [ ]:
def crop_phone(i, pad_ms=0.0):
    """Crop audio by PHN row index; pad_ms adds that much context on each side."""
    r   = phones.loc[i]
    pad = int(pad_ms / 1000 * SR)
    a, b = max(0, r.start - pad), min(len(x), r.end + pad)
    return x[a:b], r


def play_phone(i, pad_ms=0.0):
    seg, r = crop_phone(i, pad_ms)
    print(f"[{i:2d}] {r.label:4s} ({phone_class(r.label):9s})  "
          f"[{r.start}, {r.end})  {r.dur_ms:6.1f} ms  "
          f"{'+' + str(int(pad_ms)) + 'ms ctx' if pad_ms else ''}")
    display(Audio(seg, rate=SR))


# a representative spread: fricative / vowel / stop closure / stop release
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]))

print("\n---- same phones, with 20 ms of context on each side ----")
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]), pad_ms=20)

print("\n---- cropping by word works the same way ----")
for wi in [6, 7]:
    w = words.loc[wi]
    print(f"word {w.label!r}  [{w.start}, {w.end})  {w.dur_ms:.0f} ms")
    display(Audio(x[w.start:w.end], rate=SR))

With `ipywidgets` installed (`pip install ipywidgets`) the next cell gives a **dropdown**
that plays the selected phone; without it, it falls back to printing a table of choices.

In [ ]:
try:
    import ipywidgets as W

    opts = [(f"{i:2d}  {r.label:5s} {r.dur_ms:6.1f} ms  [{r.start},{r.end})", i)
            for i, r in phones.iterrows()]
    dd  = W.Dropdown(options=opts, description="phone:", layout=W.Layout(width="420px"))
    pad = W.IntSlider(value=0, min=0, max=100, step=10, description="ctx (ms):")
    outw = W.Output()

    def _on(_=None):
        with outw:
            outw.clear_output()
            play_phone(dd.value, pad_ms=pad.value)

    dd.observe(_on, names="value"); pad.observe(_on, names="value")
    display(W.VBox([W.HBox([dd, pad]), outw])); _on()
except ImportError:
    print("ipywidgets not installed -- pick manually with play_phone(i). Available phones:\n")
    print(phones.assign(cls=phones.label.map(phone_class))
                [["label", "cls", "start", "end", "dur_ms"]].to_string())
    print("\ne.g.  play_phone(13, pad_ms=20)")

### D.2 Sample-to-model-frame alignment — 完成对齐

Goal: translate `.PHN`'s `[start_sample, end_sample)` into exact encoder frame indices for
**HuBERT base**.

#### (D.2.1) Feed in one 16 kHz waveform

Preprocess with the checkpoint's **own** `AutoFeatureExtractor` — the `do_normalize` and
`return_attention_mask` policies differ per checkpoint, and hand-rolled normalisation fails
silently.

In [ ]:
MODEL_KEY, HF_ID = "hubert-base", "facebook/hubert-base-ls960"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")
words    = read_timit_seg(UTT_STEM + ".WRD")
assert sr == 16000, f"model expects 16 kHz, got {sr}"

display(describe_frontend(HF_ID))

fe, model, cfg = load_model(HF_ID)
inputs = fe(x, sampling_rate=SR, return_tensors="pt")
print("feature extractor output:", {k: tuple(v.shape) for k, v in inputs.items()})
print(f"raw waveform      mean={x.mean():+.5f}  std={x.std():.5f}")
iv = inputs["input_values"][0].numpy()
print(f"fed to the model  mean={iv.mean():+.5f}  std={iv.std():.5f}   "
      f"(do_normalize={fe.do_normalize})")

#### (D.2.2) Print the conv output and every hidden-state shape

`model.feature_extractor` *is* the 7-layer conv stack; it returns `(B, 512, T)` — note that
it is **channel-first**. `output_hidden_states=True` yields **13** tensors:
`hidden_states[0]` is the input to the Transformer, `hidden_states[i]` is the output of
Transformer layer `i`, and `hidden_states[12] is last_hidden_state`.

In [ ]:
R  = forward_frozen(HF_ID, x)
hs = R["hidden_states"]
T  = R["T"]

print(f"waveform                       {tuple(R['inputs']['input_values'].shape)}   ({len(x)} samples)")
print(f"conv feature_extractor output  {tuple(R['conv'].shape)}   (B, C=512, T) channel-first")
print(f"conv transposed                {tuple(R['conv'].transpose(1,2).shape)}   (B, T, C)")
if getattr(R["out"], "extract_features", None) is not None:
    print(f"outputs.extract_features       {tuple(R['out'].extract_features.shape)}   (layer-normed conv features)")
print(f"last_hidden_state              {tuple(R['out'].last_hidden_state.shape)}")
print(f"\nhidden_states: {len(hs)} tensors (= 1 + num_hidden_layers = 1 + {cfg.num_hidden_layers})")
for i, h in enumerate(hs):
    tag = "after conv projection / before Transformer" if i == 0 else f"output of Transformer layer {i}"
    star = "   <- last_hidden_state" if i == len(hs) - 1 else ""
    print(f"   hidden_states[{i:2d}]  {tuple(h.shape)}   {tag}{star}")
print("\nhidden_states[-1] is last_hidden_state ?",
      torch.equal(hs[-1], R["out"].last_hidden_state))

# length: hand-derived formula == HF's internal helper == the actual output
n = len(x)
print(f"\nlength check  n={n} samples")
print(f"   hand-derived conv_out_len(n)               = {conv_out_len(n)}")
print(f"   HF _get_feat_extract_output_lengths(n)     = {int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])}")
print(f"   actual T                                   = {T}")
assert conv_out_len(n) == T == int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])
print(f"\nLast frame's receptive field ends at {HOP*(T-1)+WIN}; the waveform is {n} samples "
      f"-> the final {n-(HOP*(T-1)+WIN)} samples are dropped, since the convs do not pad")

#### (D.2.3) Map PHN's `[start_sample, end_sample)` onto encoder frames

Two mappings, for two different purposes:

* **overlap** — the receptive field intersects the phone. Permissive; frames get shared
  between phones. Use it to ask *"which frames ever saw this phone?"*
* **centre** — the receptive-field **centre** falls inside the phone. Strict, and it forms a
  **partition** of the frames (each frame belongs to exactly one phone). This is the one you
  must use to build per-frame classification labels.

In [ ]:
rows = []
for r in phones.itertuples():
    fo = frames_overlapping(r.start, r.end, T)
    fc = frames_by_center(r.start, r.end, T)
    rows.append((r.Index, r.label, phone_class(r.label), r.start, r.end, round(r.dur_ms, 1),
                 f"[{fo[0]},{fo[-1]}]" if fo else "-", len(fo),
                 f"[{fc[0]},{fc[-1]}]" if fc else "-", len(fc)))
align = pd.DataFrame(rows, columns=["i", "phone", "class", "start", "end", "dur_ms",
                                    "overlap_frames", "n_overlap",
                                    "center_frames", "n_center"])
display(align)

print(f"centre-mapped frames sum to {align.n_center.sum()};  T = {T}   "
      f"-> exactly a partition: {align.n_center.sum() == T}")
print(f"overlap-mapped frames sum to {align.n_overlap.sum()} (> T, boundary frames double-counted)")

lost = align[align.n_center == 0]
print(f"\nPhones that receive NO centre frame at all "
      f"(anything shorter than the 20 ms hop can vanish entirely):")
display(lost[["i", "phone", "class", "start", "end", "dur_ms", "overlap_frames"]])
print("This is the intrinsic cost of a 20 ms frame shift: in per-frame phone labels, "
      "these phones simply disappear.")

#### (D.2.4) Check whether a frame's centre really falls inside the target phone

Taking the `/s/` in `suit`, walk every overlap frame and print its receptive field, its
centre, whether that centre is inside the phone, and how much of the receptive field the
phone actually covers.

In [ ]:
TARGET = "s"                                    # try any other phone here
ti = int(align[align.phone == TARGET].i.iloc[0])
tr = phones.loc[ti]
print(f"target phone: [{ti}] {tr.label}  [{tr.start}, {tr.end})  {tr.dur_ms:.1f} ms "
      f"= [{tr.t_start:.4f}, {tr.t_end:.4f}] s\n")

chk = []
for t in frames_overlapping(tr.start, tr.end, T):
    a, b = frame_span(t)
    c    = frame_center(t)
    inside = bool(tr.start <= c < tr.end)
    ovrlp_num   = max(0, min(b, tr.end) - max(a, tr.start))
    chk.append((t, a, b, c, round(c / SR, 4), inside, ovrlp_num, round(100 * ovrlp_num / WIN, 1)))
chk = pd.DataFrame(chk, columns=["frame", "rf_start", "rf_end", "center", "center_s",
                                 "center_inside_phone", "overlap_samples", "overlap_%"])
display(chk)
print("Look at the first and last rows: their receptive fields do intersect /s/, but their "
      "centres already lie outside it, so the centre mapping excludes them. That is "
      "precisely where the two mappings part ways.")

# the global per-frame label table
ftab = frame_label_table(T, phones)
display(ftab.head(10))
print("...")
display(ftab.tail(5))
assert (ftab.phone != "<none>").all(), "some frame centre falls outside PHN coverage"

**Empirical check.** The `[320t, 320t+400)` span above came out of pure convolution
arithmetic. Is it actually true? Replace the normalisation layers in the conv stack with
`Identity`, perturb **a single sample**, and see which frames change.

In [ ]:
probe_sample = 2440                       # an arbitrary sample
pred = [t for t in range(T) if frame_span(t)[0] <= probe_sample < frame_span(t)[1]]

# (a) normalisation removed -> pure conv arithmetic, should match the formula exactly
m2, n_repl = copy.deepcopy(model), 0
for mod in m2.feature_extractor.modules():
    for cn, ch in list(mod.named_children()):
        if isinstance(ch, (nn.GroupNorm, nn.LayerNorm, nn.BatchNorm1d)):
            setattr(mod, cn, nn.Identity()); n_repl += 1

xb = torch.randn(1, len(x)) * 0.05
xp = xb.clone(); xp[0, probe_sample] += 1.0
with torch.no_grad():
    d_pure = (m2.feature_extractor(xp) - m2.feature_extractor(xb)).abs().sum(1)[0]
got = torch.nonzero(d_pure > 1e-5).flatten().tolist()
print(f"replaced {n_repl} normalisation layer(s) in the conv stack")
print(f"perturbing sample {probe_sample} -> frames affected {got}   formula predicts {pred}   match: {got == pred}")

# (b) normalisation kept -> measure how much global information it leaks
with torch.no_grad():
    d_real = (model.feature_extractor(xp) - model.feature_extractor(xb)).abs().sum(1)[0]
mask = torch.ones(T, dtype=bool); mask[pred] = False
inside, outside = d_real[pred].sum().item(), d_real[mask].sum().item()
print(f"\nwith normalisation kept: change inside the receptive field {inside:.4f} / "
      f"outside {outside:.4f} = {inside/max(outside,1e-12):.1f}x")
print(f"normalisation in the conv stack: {[type(m).__name__ for m in model.feature_extractor.modules() if isinstance(m,(nn.GroupNorm,nn.LayerNorm))]}")
print("""
Takeaways:
  * Under pure convolution arithmetic the receptive field [320t, 320t+400) is EXACT --
    it matches for all five models.
  * But GroupNorm computes its statistics along the TIME AXIS, so every frame ends up
    dependent on the whole utterance. Note how small the ratio above is: for the four
    GroupNorm checkpoints it lands around 0.5x-1.5x, i.e. the normalisation pathway carries
    as much of the perturbation as the local receptive field does. data2vec instead uses
    per-frame LayerNorm (7 of them) and does not leak at all -- its ratio is ~1e13.
  * More important still: all of this holds only for the **conv output / hidden_states[0]**.
    Past the first self-attention block, every frame of hidden_states[i>=1] can in principle
    see the entire utterance -- the frame-to-phone alignment is a **positional**
    correspondence, not a bound on where the information came from.
""")

#### (D.2.5) Visualise the hidden-state rows for a phone span

Take the centre frames of `/s/` above and plot `hidden_states[layer][0, t0:t1, :]` directly
as a heatmap.

In [ ]:
fc = frames_by_center(tr.start, tr.end, T)
t0, t1 = fc[0], fc[-1] + 1
print(f"phone {tr.label!r} -> hidden_states[L][0, {t0}:{t1}, :]  = {t1-t0} frames x {cfg.hidden_size} dims")

show = [0, 1, 6, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.0 * len(show), 3.6))
for ax, li in zip(axes, show):
    blk = hs[li][0, t0:t1].numpy()
    v   = np.abs(blk).max()
    im  = ax.imshow(blk, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                    interpolation="nearest")
    ax.set_title(f"hidden_states[{li}]\n{tr.label} rows {t0}:{t1}", fontsize=10)
    ax.set_xlabel("hidden dim (768)")
    ax.set_yticks(range(t1 - t0)); ax.set_yticklabels(range(t0, t1), fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel("encoder frame")
    plt.colorbar(im, ax=ax, fraction=0.035)
fig.suptitle(f"HuBERT base - hidden-state rows for /{tr.label}/", y=1.04)
fig.tight_layout(); plt.show()

# whole utterance, to confirm the alignment lands where it should
fig, axes = plt.subplots(2, 1, figsize=(16, 6.4),
                         gridspec_kw={"height_ratios": [1, 1.5]})
plot_wave_with_phones(x, phones, ax=axes[0], title=f"HuBERT base - waveform + PHN")
axes[0].axvspan(tr.t_start, tr.t_end, color="k", alpha=0.18, zorder=1)

H = hs[6][0].numpy()
v = np.percentile(np.abs(H), 99)
axes[1].imshow(H.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[0, T * HOP / SR, 0, H.shape[1]], interpolation="nearest")
for r in phones.itertuples():
    axes[1].axvline(r.t_start, color="k", lw=0.5, alpha=0.45)
axes[1].axvspan(tr.t_start, tr.t_end, facecolor="none", edgecolor="lime", lw=2.2)
axes[1].set_xlim(0, len(x) / SR)
axes[1].set_title("hidden_states[6]  (transposed: dim x frame); black = PHN boundaries, "
                  "green box = target phone")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("hidden dim")
fig.tight_layout(); plt.show()

### D.3 Probe the frozen model's layers — 探索冻结模型的各层表示

#### (D.3.1) Freeze and `eval()`

`eval()` turns off dropout and SpecAugment (`apply_spec_augment` only fires while training);
`requires_grad = False` cuts the gradients. The two are **independent** — do both.

In [ ]:
MODEL_KEY, HF_ID = "hubert-base", "facebook/hubert-base-ls960"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")

fe, model, cfg = load_model(HF_ID)

model.eval()
for p in model.parameters():
    p.requires_grad = False

n_all   = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"HuBERT base  ({HF_ID})")
print(f"  model.training                  = {model.training}   (False -> dropout / SpecAugment off)")
print(f"  total parameters                = {n_all/1e6:.1f} M")
print(f"  parameters with requires_grad   = {n_train}   -> fully frozen: {n_train == 0}")
print(f"  config.apply_spec_augment       = {getattr(cfg, 'apply_spec_augment', None)} "
      f"(only takes effect while training=True)")

#### (D.3.2) Forward pass with `output_hidden_states=True`

In [ ]:
with torch.no_grad():
    inputs = fe(x, sampling_rate=SR, return_tensors="pt")
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

hs = outputs.hidden_states
T  = outputs.last_hidden_state.shape[1]
print("outputs fields:", list(outputs.keys()))
print(f"hidden_states: {len(hs)} x {tuple(hs[0].shape)}   T={T} frames "
      f"({T*HOP/SR:.3f} s @ {HOP/SR*1000:.0f} ms/frame)")
print("gradients detached (requires_grad):", outputs.last_hidden_state.requires_grad)

ftab          = frame_label_table(T, phones)
frame_phones  = ftab.phone.values
frame_classes = ftab["class"].values
print("\nper-frame label distribution:")
display(ftab["class"].value_counts().to_frame("n_frames").T)

#### (D.3.3) Per-layer statistics

The scale of each layer. The final layer's norm often blows up noticeably — which bears
directly on *which* layer to take as downstream features, and whether to layer-norm first.

In [ ]:
st = layer_stats(hs)
display(st.style.format({"mean": "{:+.5f}", "std": "{:.4f}", "mean_row_norm": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(st.layer, st.mean_row_norm, "o-", color="#4C78A8")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("mean ||h_t||")
axes[0].set_title(f"HuBERT base - mean L2 norm per frame vector"); axes[0].grid(alpha=0.3)
axes[1].plot(st.layer, st["std"], "o-", color="#E45756")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("std")
axes[1].set_title("activation standard deviation"); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

#### (D.3.4) How similar are the layers: cosine and linear CKA

CKA is invariant to rotation and scaling, which makes it the standard choice for comparing
representations; the cosine matrix shows more directly which layers barely change anything.

In [ ]:
Mcos = layer_cosine(hs)
Mcka = layer_cka(hs)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, M, name in zip(axes, [Mcos, Mcka], ["per-frame cosine (mean-centred)", "linear CKA"]):
    im = ax.imshow(M, cmap="viridis", vmin=np.min([Mcos.min(), 0]), vmax=1.0)
    ax.set_title(f"{name}"); ax.set_xlabel("layer"); ax.set_ylabel("layer")
    ax.set_xticks(range(len(hs))); ax.set_yticks(range(len(hs)))
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"HuBERT base - representation similarity between layers", y=1.01)
fig.tight_layout(); plt.show()

adj = pd.DataFrame({"layer_pair": [f"L{i}->L{i+1}" for i in range(len(hs)-1)],
                    "cosine": np.diag(Mcos, 1).round(4),
                    "CKA":    np.diag(Mcka, 1).round(4)})
display(adj.T)
print("The closer an adjacent-layer CKA is to 1, the less that layer changed; the lowest "
      "pairs mark where the representation shifts most sharply.")

#### (D.3.5) Phonetic information layer by layer

Three angles, ordered from **most trustworthy** to **most in need of caution**:

1. **Adjacent-frame contrast** (trustworthy): cosine similarity of adjacent frames within a
   phone vs across a phone boundary. ~144 adjacent pairs is enough statistical power on a
   single utterance, and a positive gap independently confirms the alignment is correct.
2. **Self-similarity matrix** (qualitative): the `T×T` frame-by-frame cosine matrix, where
   phone segments show up as blocks along the diagonal.
3. **Broad-class separability** (treat with caution): silhouette plus a 5-fold linear probe.
   With **one utterance, 138 frames, 6 classes** this is very noisy — what is being
   demonstrated is the **method**. Do not read the layer ordering as reproducing published
   curves; that needs hundreds of utterances.

In [ ]:
bc = boundary_contrast(hs, frame_phones)
display(bc.style.format({"within_phone": "{:.4f}", "across_boundary": "{:.4f}",
                         "gap": "{:+.4f}", "boundary_AUC": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(bc.layer, bc.within_phone,    "o-", label="within phone",    color="#54A24B")
axes[0].plot(bc.layer, bc.across_boundary, "o-", label="across boundary", color="#E45756")
axes[0].fill_between(bc.layer, bc.across_boundary, bc.within_phone, alpha=0.15, color="#54A24B")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("adjacent-frame cosine")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title(f"HuBERT base - adjacent-frame similarity")
axes[1].plot(bc.layer, bc.boundary_AUC, "o-", color="#4C78A8")
axes[1].axhline(0.5, ls="--", c="gray", lw=1)
axes[1].set_xlabel("layer"); axes[1].set_ylabel("AUC")
axes[1].set_title("phone-boundary detection AUC from adjacent-frame similarity")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"gap positive at every layer: {bool((bc.gap > 0).all())}  -> every layer retains phone "
      f"boundary information, and the alignment checks out")
print(f"largest gap: L{int(bc.gap.idxmax())} ({bc.gap.max():+.4f}), "
      f"smallest: L{int(bc.gap.idxmin())} ({bc.gap.min():+.4f})")

In [ ]:
# Self-similarity: T x T frame-by-frame cosine; phone segments appear as diagonal blocks
show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.3))
bnd = [r.t_start for r in phones.itertuples()]
for ax, li in zip(axes, show):
    v = Fn.normalize(hs[li][0].float(), dim=-1)
    S = (v @ v.T).numpy()
    ax.imshow(S, cmap="magma", origin="lower", vmin=np.percentile(S, 2), vmax=1.0,
              extent=[0, T * HOP / SR, 0, T * HOP / SR], interpolation="nearest")
    for b in bnd:
        ax.axvline(b, color="#66ccff", lw=0.4, alpha=0.55)
        ax.axhline(b, color="#66ccff", lw=0.4, alpha=0.55)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xlabel("time (s)")
    if ax is axes[0]:
        ax.set_ylabel("time (s)")
fig.suptitle(f"HuBERT base - frame-by-frame self-similarity (blue lines = PHN boundaries)", y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
sep, keep, yk = phone_separability(hs, frame_classes, min_count=5)
display(sep.style.format({"silhouette_cos": "{:+.4f}", "probe_acc_5fold": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(sep.layer, sep.silhouette_cos, "o-", color="#B279A2")
axes[0].axhline(0, ls="--", c="gray", lw=1)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("silhouette (cosine)")
axes[0].set_title("cluster geometry of broad phone classes"); axes[0].grid(alpha=0.3)
axes[1].plot(sep.layer, sep.probe_acc_5fold, "o-", color="#F58518")
axes[1].axhline(pd.Series(yk).value_counts(normalize=True).max(), ls="--", c="gray", lw=1,
                label="majority-class baseline")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("5-fold accuracy")
axes[1].set_title("linear probe (broad phone class)"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle(f"HuBERT base - per-layer phone separability "
             f"(ONE utterance, {keep.sum()} frames - very noisy)", y=1.05)
fig.tight_layout(); plt.show()

print(f"frames used: {keep.sum()} / {T}, classes: {sorted(set(yk))}")
print(f"best silhouette: L{int(sep.silhouette_cos.idxmax())}   "
      f"best probe: L{int(sep.probe_acc_5fold.idxmax())}")
print("""
WARNING: with a single utterance the layer ordering of these two curves is NOT reliable and
should not be compared against published results. To get a meaningful curve, wrap
forward_frozen over a few hundred utterances, accumulate per-frame features and labels until
you have ~1e5 frames, and split train/test by speaker.
""")

In [ ]:
# PCA: project each frame to 2-D, colour by broad phone class
from sklearn.decomposition import PCA

show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.0))
for ax, li in zip(axes, show):
    Z = PCA(n_components=2, random_state=0).fit_transform(hs[li][0].float().numpy())
    for c in sorted(set(frame_classes)):
        m = frame_classes == c
        ax.scatter(Z[m, 0], Z[m, 1], s=20, c=CLASS_COLOR[c], label=c,
                   edgecolors="none", alpha=0.85)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="best", frameon=False)
fig.suptitle(f"HuBERT base - PCA of per-frame representations (colour = broad phone class)", y=1.03)
fig.tight_layout(); plt.show()

---

# E. data2vec audio base — `facebook/data2vec-audio-base`

`facebook/data2vec-audio-base` (LibriSpeech 960h), self-distillation: the student predicts the
EMA teacher's **top-k-layer-averaged** continuous representation — no discrete codebook.

* Front end: `do_normalize=True`, `return_attention_mask=True`.
* **The most substantive structural difference from the other four**: its conv stack uses
  **per-layer LayerNorm** (7 of them) rather than a single GroupNorm at the front. LayerNorm
  normalises across channels and never across time, so its conv receptive field
  `[320t, 320t+400)` is **strictly local** — visible in the empirical check in E.2.4.

### E.1 Read one utterance — 看懂一条数据

> This task is model-independent (pure TIMIT parsing), so its output is the same in all five
> model sections. It is repeated so that the **data2vec audio base** section stands on its own.

#### (E.1.1) Read the SPHERE audio

Print the 1024-byte NIST header field by field first, then read the waveform and cross-check
it against a hand-rolled parse.

In [ ]:
MODEL_KEY, HF_ID = "data2vec-audio-base", "facebook/data2vec-audio-base"
print(f"### data2vec audio base  (facebook/data2vec-audio-base) ###\n")

magic, header_bytes, F = read_sphere_header(UTT_STEM + ".WAV")
print(f"SPHERE magic = {magic}   header = {header_bytes} bytes")
for k, v in F.items():
    print(f"   {k:20s} {v}")

x, sr, how = read_sphere(UTT_STEM + ".WAV")
print(f"\nread via: {how}")
print(f"waveform: shape={x.shape}  dtype={x.dtype}  sr={sr}  "
      f"duration={len(x)/sr:.4f} s  range=[{x.min():.4f}, {x.max():.4f}]")

# Cross-check: manual header parse + raw PCM should match libsndfile exactly
raw = np.fromfile(UTT_STEM + ".WAV", dtype="<i2",
                  offset=header_bytes, count=F["sample_count"]).astype(np.float32) / 32768.0
print(f"manual parse == soundfile ? {np.allclose(raw, x)}   (max |diff| = {np.abs(raw-x).max():.2e})")
assert sr == 16000 and x.ndim == 1

#### (E.1.2) Read `.TXT` / `.WRD` / `.PHN`

All three are **sample indices over half-open intervals `[start, end)`**.

In [ ]:
txt_s, txt_e, transcript = read_timit_txt(UTT_STEM + ".TXT")
words  = read_timit_seg(UTT_STEM + ".WRD")
phones = read_timit_seg(UTT_STEM + ".PHN")

print(f".TXT  [{txt_s}, {txt_e})  {transcript!r}")
print(f".WRD  {len(words)} words      .PHN  {len(phones)} phones\n")

display(words)
display(phones)

print("phone classes present:", sorted(set(phones.label.map(phone_class))))

# --- three real TIMIT gotchas visible in this very utterance ---
ov = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i].end - words.iloc[i+1].start)
      for i in range(len(words)-1) if words.iloc[i].end > words.iloc[i+1].start]
gp = [(words.iloc[i].label, words.iloc[i+1].label,
       words.iloc[i+1].start - words.iloc[i].end)
      for i in range(len(words)-1) if words.iloc[i].end < words.iloc[i+1].start]
print(f"\n[gotcha 1] word spans that OVERLAP (overlap in samples): {ov}")
print(f"[gotcha 2] GAPS between words (gap in samples): {gp}")
print(f"[gotcha 3] PHN covers up to {phones.end.iloc[-1]}, but the waveform has {len(x)} "
      f"samples -> the last {len(x)-phones.end.iloc[-1]} samples are unlabelled")
print("\nBy contrast PHN itself is GAPLESS (every end == the next start):",
      bool((phones.end.values[:-1] == phones.start.values[1:]).all()))

#### (E.1.3) Play the whole utterance

In [ ]:
print(transcript)
display(Audio(x, rate=SR))

#### (E.1.4) Plot the waveform with phone boundaries and labels overlaid

Phone tier on top (colour = broad phone class), word tier in blue below. The second panel
zooms into `she had your`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8.5))
plot_wave_with_phones(x, phones, words, ax=axes[0],
                      title=f"data2vec audio base | {os.path.basename(UTT_STEM)} — {transcript}")
legend_classes(axes[0])
plot_wave_with_phones(x, phones, words, ax=axes[1], t0=0.15, t1=1.10,
                      title="zoom: 0.15 - 1.10 s  (she had your)")
fig.tight_layout()
plt.show()

#### (E.1.5) Crop a phone and listen to just that segment

Because `.PHN`'s `[start, end)` *is* a pair of array indices, cropping is literally
`x[start:end]`. Below: each phone on its own first, then the same phones with **20 ms of
context** added — many phones are near-unrecognisable in isolation, closures and stops
especially.

In [ ]:
def crop_phone(i, pad_ms=0.0):
    """Crop audio by PHN row index; pad_ms adds that much context on each side."""
    r   = phones.loc[i]
    pad = int(pad_ms / 1000 * SR)
    a, b = max(0, r.start - pad), min(len(x), r.end + pad)
    return x[a:b], r


def play_phone(i, pad_ms=0.0):
    seg, r = crop_phone(i, pad_ms)
    print(f"[{i:2d}] {r.label:4s} ({phone_class(r.label):9s})  "
          f"[{r.start}, {r.end})  {r.dur_ms:6.1f} ms  "
          f"{'+' + str(int(pad_ms)) + 'ms ctx' if pad_ms else ''}")
    display(Audio(seg, rate=SR))


# a representative spread: fricative / vowel / stop closure / stop release
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]))

print("\n---- same phones, with 20 ms of context on each side ----")
for lab in ["s", "eh", "dcl", "d"]:
    idx = phones.index[phones.label == lab]
    if len(idx):
        play_phone(int(idx[0]), pad_ms=20)

print("\n---- cropping by word works the same way ----")
for wi in [6, 7]:
    w = words.loc[wi]
    print(f"word {w.label!r}  [{w.start}, {w.end})  {w.dur_ms:.0f} ms")
    display(Audio(x[w.start:w.end], rate=SR))

With `ipywidgets` installed (`pip install ipywidgets`) the next cell gives a **dropdown**
that plays the selected phone; without it, it falls back to printing a table of choices.

In [ ]:
try:
    import ipywidgets as W

    opts = [(f"{i:2d}  {r.label:5s} {r.dur_ms:6.1f} ms  [{r.start},{r.end})", i)
            for i, r in phones.iterrows()]
    dd  = W.Dropdown(options=opts, description="phone:", layout=W.Layout(width="420px"))
    pad = W.IntSlider(value=0, min=0, max=100, step=10, description="ctx (ms):")
    outw = W.Output()

    def _on(_=None):
        with outw:
            outw.clear_output()
            play_phone(dd.value, pad_ms=pad.value)

    dd.observe(_on, names="value"); pad.observe(_on, names="value")
    display(W.VBox([W.HBox([dd, pad]), outw])); _on()
except ImportError:
    print("ipywidgets not installed -- pick manually with play_phone(i). Available phones:\n")
    print(phones.assign(cls=phones.label.map(phone_class))
                [["label", "cls", "start", "end", "dur_ms"]].to_string())
    print("\ne.g.  play_phone(13, pad_ms=20)")

### E.2 Sample-to-model-frame alignment — 完成对齐

Goal: translate `.PHN`'s `[start_sample, end_sample)` into exact encoder frame indices for
**data2vec audio base**.

#### (E.2.1) Feed in one 16 kHz waveform

Preprocess with the checkpoint's **own** `AutoFeatureExtractor` — the `do_normalize` and
`return_attention_mask` policies differ per checkpoint, and hand-rolled normalisation fails
silently.

In [ ]:
MODEL_KEY, HF_ID = "data2vec-audio-base", "facebook/data2vec-audio-base"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")
words    = read_timit_seg(UTT_STEM + ".WRD")
assert sr == 16000, f"model expects 16 kHz, got {sr}"

display(describe_frontend(HF_ID))

fe, model, cfg = load_model(HF_ID)
inputs = fe(x, sampling_rate=SR, return_tensors="pt")
print("feature extractor output:", {k: tuple(v.shape) for k, v in inputs.items()})
print(f"raw waveform      mean={x.mean():+.5f}  std={x.std():.5f}")
iv = inputs["input_values"][0].numpy()
print(f"fed to the model  mean={iv.mean():+.5f}  std={iv.std():.5f}   "
      f"(do_normalize={fe.do_normalize})")

#### (E.2.2) Print the conv output and every hidden-state shape

`model.feature_extractor` *is* the 7-layer conv stack; it returns `(B, 512, T)` — note that
it is **channel-first**. `output_hidden_states=True` yields **13** tensors:
`hidden_states[0]` is the input to the Transformer, `hidden_states[i]` is the output of
Transformer layer `i`, and `hidden_states[12] is last_hidden_state`.

In [ ]:
R  = forward_frozen(HF_ID, x)
hs = R["hidden_states"]
T  = R["T"]

print(f"waveform                       {tuple(R['inputs']['input_values'].shape)}   ({len(x)} samples)")
print(f"conv feature_extractor output  {tuple(R['conv'].shape)}   (B, C=512, T) channel-first")
print(f"conv transposed                {tuple(R['conv'].transpose(1,2).shape)}   (B, T, C)")
if getattr(R["out"], "extract_features", None) is not None:
    print(f"outputs.extract_features       {tuple(R['out'].extract_features.shape)}   (layer-normed conv features)")
print(f"last_hidden_state              {tuple(R['out'].last_hidden_state.shape)}")
print(f"\nhidden_states: {len(hs)} tensors (= 1 + num_hidden_layers = 1 + {cfg.num_hidden_layers})")
for i, h in enumerate(hs):
    tag = "after conv projection / before Transformer" if i == 0 else f"output of Transformer layer {i}"
    star = "   <- last_hidden_state" if i == len(hs) - 1 else ""
    print(f"   hidden_states[{i:2d}]  {tuple(h.shape)}   {tag}{star}")
print("\nhidden_states[-1] is last_hidden_state ?",
      torch.equal(hs[-1], R["out"].last_hidden_state))

# length: hand-derived formula == HF's internal helper == the actual output
n = len(x)
print(f"\nlength check  n={n} samples")
print(f"   hand-derived conv_out_len(n)               = {conv_out_len(n)}")
print(f"   HF _get_feat_extract_output_lengths(n)     = {int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])}")
print(f"   actual T                                   = {T}")
assert conv_out_len(n) == T == int(model._get_feat_extract_output_lengths(torch.tensor([n]))[0])
print(f"\nLast frame's receptive field ends at {HOP*(T-1)+WIN}; the waveform is {n} samples "
      f"-> the final {n-(HOP*(T-1)+WIN)} samples are dropped, since the convs do not pad")

#### (E.2.3) Map PHN's `[start_sample, end_sample)` onto encoder frames

Two mappings, for two different purposes:

* **overlap** — the receptive field intersects the phone. Permissive; frames get shared
  between phones. Use it to ask *"which frames ever saw this phone?"*
* **centre** — the receptive-field **centre** falls inside the phone. Strict, and it forms a
  **partition** of the frames (each frame belongs to exactly one phone). This is the one you
  must use to build per-frame classification labels.

In [ ]:
rows = []
for r in phones.itertuples():
    fo = frames_overlapping(r.start, r.end, T)
    fc = frames_by_center(r.start, r.end, T)
    rows.append((r.Index, r.label, phone_class(r.label), r.start, r.end, round(r.dur_ms, 1),
                 f"[{fo[0]},{fo[-1]}]" if fo else "-", len(fo),
                 f"[{fc[0]},{fc[-1]}]" if fc else "-", len(fc)))
align = pd.DataFrame(rows, columns=["i", "phone", "class", "start", "end", "dur_ms",
                                    "overlap_frames", "n_overlap",
                                    "center_frames", "n_center"])
display(align)

print(f"centre-mapped frames sum to {align.n_center.sum()};  T = {T}   "
      f"-> exactly a partition: {align.n_center.sum() == T}")
print(f"overlap-mapped frames sum to {align.n_overlap.sum()} (> T, boundary frames double-counted)")

lost = align[align.n_center == 0]
print(f"\nPhones that receive NO centre frame at all "
      f"(anything shorter than the 20 ms hop can vanish entirely):")
display(lost[["i", "phone", "class", "start", "end", "dur_ms", "overlap_frames"]])
print("This is the intrinsic cost of a 20 ms frame shift: in per-frame phone labels, "
      "these phones simply disappear.")

#### (E.2.4) Check whether a frame's centre really falls inside the target phone

Taking the `/s/` in `suit`, walk every overlap frame and print its receptive field, its
centre, whether that centre is inside the phone, and how much of the receptive field the
phone actually covers.

In [ ]:
TARGET = "s"                                    # try any other phone here
ti = int(align[align.phone == TARGET].i.iloc[0])
tr = phones.loc[ti]
print(f"target phone: [{ti}] {tr.label}  [{tr.start}, {tr.end})  {tr.dur_ms:.1f} ms "
      f"= [{tr.t_start:.4f}, {tr.t_end:.4f}] s\n")

chk = []
for t in frames_overlapping(tr.start, tr.end, T):
    a, b = frame_span(t)
    c    = frame_center(t)
    inside = bool(tr.start <= c < tr.end)
    ov   = max(0, min(b, tr.end) - max(a, tr.start))
    chk.append((t, a, b, c, round(c / SR, 4), inside, ov, round(100 * ov / WIN, 1)))
chk = pd.DataFrame(chk, columns=["frame", "rf_start", "rf_end", "center", "center_s",
                                 "center_inside_phone", "overlap_samples", "overlap_%"])
display(chk)
print("Look at the first and last rows: their receptive fields do intersect /s/, but their "
      "centres already lie outside it, so the centre mapping excludes them. That is "
      "precisely where the two mappings part ways.")

# the global per-frame label table
ftab = frame_label_table(T, phones)
display(ftab.head(10))
print("...")
display(ftab.tail(5))
assert (ftab.phone != "<none>").all(), "some frame centre falls outside PHN coverage"

**Empirical check.** The `[320t, 320t+400)` span above came out of pure convolution
arithmetic. Is it actually true? Replace the normalisation layers in the conv stack with
`Identity`, perturb **a single sample**, and see which frames change.

In [ ]:
probe_sample = 2440                       # an arbitrary sample
pred = [t for t in range(T) if frame_span(t)[0] <= probe_sample < frame_span(t)[1]]

# (a) normalisation removed -> pure conv arithmetic, should match the formula exactly
m2, n_repl = copy.deepcopy(model), 0
for mod in m2.feature_extractor.modules():
    for cn, ch in list(mod.named_children()):
        if isinstance(ch, (nn.GroupNorm, nn.LayerNorm, nn.BatchNorm1d)):
            setattr(mod, cn, nn.Identity()); n_repl += 1

xb = torch.randn(1, len(x)) * 0.05
xp = xb.clone(); xp[0, probe_sample] += 1.0
with torch.no_grad():
    d_pure = (m2.feature_extractor(xp) - m2.feature_extractor(xb)).abs().sum(1)[0]
got = torch.nonzero(d_pure > 1e-5).flatten().tolist()
print(f"replaced {n_repl} normalisation layer(s) in the conv stack")
print(f"perturbing sample {probe_sample} -> frames affected {got}   formula predicts {pred}   match: {got == pred}")

# (b) normalisation kept -> measure how much global information it leaks
with torch.no_grad():
    d_real = (model.feature_extractor(xp) - model.feature_extractor(xb)).abs().sum(1)[0]
mask = torch.ones(T, dtype=bool); mask[pred] = False
inside, outside = d_real[pred].sum().item(), d_real[mask].sum().item()
print(f"\nwith normalisation kept: change inside the receptive field {inside:.4f} / "
      f"outside {outside:.4f} = {inside/max(outside,1e-12):.1f}x")
print(f"normalisation in the conv stack: {[type(m).__name__ for m in model.feature_extractor.modules() if isinstance(m,(nn.GroupNorm,nn.LayerNorm))]}")
print("""
Takeaways:
  * Under pure convolution arithmetic the receptive field [320t, 320t+400) is EXACT --
    it matches for all five models.
  * But GroupNorm computes its statistics along the TIME AXIS, so every frame ends up
    dependent on the whole utterance. Note how small the ratio above is: for the four
    GroupNorm checkpoints it lands around 0.5x-1.5x, i.e. the normalisation pathway carries
    as much of the perturbation as the local receptive field does. data2vec instead uses
    per-frame LayerNorm (7 of them) and does not leak at all -- its ratio is ~1e13.
  * More important still: all of this holds only for the **conv output / hidden_states[0]**.
    Past the first self-attention block, every frame of hidden_states[i>=1] can in principle
    see the entire utterance -- the frame-to-phone alignment is a **positional**
    correspondence, not a bound on where the information came from.
""")

#### (E.2.5) Visualise the hidden-state rows for a phone span

Take the centre frames of `/s/` above and plot `hidden_states[layer][0, t0:t1, :]` directly
as a heatmap.

In [ ]:
fc = frames_by_center(tr.start, tr.end, T)
t0, t1 = fc[0], fc[-1] + 1
print(f"phone {tr.label!r} -> hidden_states[L][0, {t0}:{t1}, :]  = {t1-t0} frames x {cfg.hidden_size} dims")

show = [0, 1, 6, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.0 * len(show), 3.6))
for ax, li in zip(axes, show):
    blk = hs[li][0, t0:t1].numpy()
    v   = np.abs(blk).max()
    im  = ax.imshow(blk, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                    interpolation="nearest")
    ax.set_title(f"hidden_states[{li}]\n{tr.label} rows {t0}:{t1}", fontsize=10)
    ax.set_xlabel("hidden dim (768)")
    ax.set_yticks(range(t1 - t0)); ax.set_yticklabels(range(t0, t1), fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel("encoder frame")
    plt.colorbar(im, ax=ax, fraction=0.035)
fig.suptitle(f"data2vec audio base - hidden-state rows for /{tr.label}/", y=1.04)
fig.tight_layout(); plt.show()

# whole utterance, to confirm the alignment lands where it should
fig, axes = plt.subplots(2, 1, figsize=(16, 6.4),
                         gridspec_kw={"height_ratios": [1, 1.5]})
plot_wave_with_phones(x, phones, ax=axes[0], title=f"data2vec audio base - waveform + PHN")
axes[0].axvspan(tr.t_start, tr.t_end, color="k", alpha=0.18, zorder=1)

H = hs[6][0].numpy()
v = np.percentile(np.abs(H), 99)
axes[1].imshow(H.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[0, T * HOP / SR, 0, H.shape[1]], interpolation="nearest")
for r in phones.itertuples():
    axes[1].axvline(r.t_start, color="k", lw=0.5, alpha=0.45)
axes[1].axvspan(tr.t_start, tr.t_end, facecolor="none", edgecolor="lime", lw=2.2)
axes[1].set_xlim(0, len(x) / SR)
axes[1].set_title("hidden_states[6]  (transposed: dim x frame); black = PHN boundaries, "
                  "green box = target phone")
axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("hidden dim")
fig.tight_layout(); plt.show()

### E.3 Probe the frozen model's layers — 探索冻结模型的各层表示

#### (E.3.1) Freeze and `eval()`

`eval()` turns off dropout and SpecAugment (`apply_spec_augment` only fires while training);
`requires_grad = False` cuts the gradients. The two are **independent** — do both.

In [ ]:
MODEL_KEY, HF_ID = "data2vec-audio-base", "facebook/data2vec-audio-base"
x, sr, _ = read_sphere(UTT_STEM + ".WAV")
phones   = read_timit_seg(UTT_STEM + ".PHN")

fe, model, cfg = load_model(HF_ID)

model.eval()
for p in model.parameters():
    p.requires_grad = False

n_all   = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"data2vec audio base  ({HF_ID})")
print(f"  model.training                  = {model.training}   (False -> dropout / SpecAugment off)")
print(f"  total parameters                = {n_all/1e6:.1f} M")
print(f"  parameters with requires_grad   = {n_train}   -> fully frozen: {n_train == 0}")
print(f"  config.apply_spec_augment       = {getattr(cfg, 'apply_spec_augment', None)} "
      f"(only takes effect while training=True)")

#### (E.3.2) Forward pass with `output_hidden_states=True`

In [ ]:
with torch.no_grad():
    inputs = fe(x, sampling_rate=SR, return_tensors="pt")
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
    )

hs = outputs.hidden_states
T  = outputs.last_hidden_state.shape[1]
print("outputs fields:", list(outputs.keys()))
print(f"hidden_states: {len(hs)} x {tuple(hs[0].shape)}   T={T} frames "
      f"({T*HOP/SR:.3f} s @ {HOP/SR*1000:.0f} ms/frame)")
print("gradients detached (requires_grad):", outputs.last_hidden_state.requires_grad)

ftab          = frame_label_table(T, phones)
frame_phones  = ftab.phone.values
frame_classes = ftab["class"].values
print("\nper-frame label distribution:")
display(ftab["class"].value_counts().to_frame("n_frames").T)

#### (E.3.3) Per-layer statistics

The scale of each layer. The final layer's norm often blows up noticeably — which bears
directly on *which* layer to take as downstream features, and whether to layer-norm first.

In [ ]:
st = layer_stats(hs)
display(st.style.format({"mean": "{:+.5f}", "std": "{:.4f}", "mean_row_norm": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
axes[0].plot(st.layer, st.mean_row_norm, "o-", color="#4C78A8")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("mean ||h_t||")
axes[0].set_title(f"data2vec audio base - mean L2 norm per frame vector"); axes[0].grid(alpha=0.3)
axes[1].plot(st.layer, st["std"], "o-", color="#E45756")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("std")
axes[1].set_title("activation standard deviation"); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

#### (E.3.4) How similar are the layers: cosine and linear CKA

CKA is invariant to rotation and scaling, which makes it the standard choice for comparing
representations; the cosine matrix shows more directly which layers barely change anything.

In [ ]:
Mcos = layer_cosine(hs)
Mcka = layer_cka(hs)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
for ax, M, name in zip(axes, [Mcos, Mcka], ["per-frame cosine (mean-centred)", "linear CKA"]):
    im = ax.imshow(M, cmap="viridis", vmin=np.min([Mcos.min(), 0]), vmax=1.0)
    ax.set_title(f"{name}"); ax.set_xlabel("layer"); ax.set_ylabel("layer")
    ax.set_xticks(range(len(hs))); ax.set_yticks(range(len(hs)))
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"data2vec audio base - representation similarity between layers", y=1.01)
fig.tight_layout(); plt.show()

adj = pd.DataFrame({"layer_pair": [f"L{i}->L{i+1}" for i in range(len(hs)-1)],
                    "cosine": np.diag(Mcos, 1).round(4),
                    "CKA":    np.diag(Mcka, 1).round(4)})
display(adj.T)
print("The closer an adjacent-layer CKA is to 1, the less that layer changed; the lowest "
      "pairs mark where the representation shifts most sharply.")

#### (E.3.5) Phonetic information layer by layer

Three angles, ordered from **most trustworthy** to **most in need of caution**:

1. **Adjacent-frame contrast** (trustworthy): cosine similarity of adjacent frames within a
   phone vs across a phone boundary. ~144 adjacent pairs is enough statistical power on a
   single utterance, and a positive gap independently confirms the alignment is correct.
2. **Self-similarity matrix** (qualitative): the `T×T` frame-by-frame cosine matrix, where
   phone segments show up as blocks along the diagonal.
3. **Broad-class separability** (treat with caution): silhouette plus a 5-fold linear probe.
   With **one utterance, 138 frames, 6 classes** this is very noisy — what is being
   demonstrated is the **method**. Do not read the layer ordering as reproducing published
   curves; that needs hundreds of utterances.

In [ ]:
bc = boundary_contrast(hs, frame_phones)
display(bc.style.format({"within_phone": "{:.4f}", "across_boundary": "{:.4f}",
                         "gap": "{:+.4f}", "boundary_AUC": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(bc.layer, bc.within_phone,    "o-", label="within phone",    color="#54A24B")
axes[0].plot(bc.layer, bc.across_boundary, "o-", label="across boundary", color="#E45756")
axes[0].fill_between(bc.layer, bc.across_boundary, bc.within_phone, alpha=0.15, color="#54A24B")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("adjacent-frame cosine")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title(f"data2vec audio base - adjacent-frame similarity")
axes[1].plot(bc.layer, bc.boundary_AUC, "o-", color="#4C78A8")
axes[1].axhline(0.5, ls="--", c="gray", lw=1)
axes[1].set_xlabel("layer"); axes[1].set_ylabel("AUC")
axes[1].set_title("phone-boundary detection AUC from adjacent-frame similarity")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"gap positive at every layer: {bool((bc.gap > 0).all())}  -> every layer retains phone "
      f"boundary information, and the alignment checks out")
print(f"largest gap: L{int(bc.gap.idxmax())} ({bc.gap.max():+.4f}), "
      f"smallest: L{int(bc.gap.idxmin())} ({bc.gap.min():+.4f})")

In [ ]:
# Self-similarity: T x T frame-by-frame cosine; phone segments appear as diagonal blocks
show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.3))
bnd = [r.t_start for r in phones.itertuples()]
for ax, li in zip(axes, show):
    v = Fn.normalize(hs[li][0].float(), dim=-1)
    S = (v @ v.T).numpy()
    ax.imshow(S, cmap="magma", origin="lower", vmin=np.percentile(S, 2), vmax=1.0,
              extent=[0, T * HOP / SR, 0, T * HOP / SR], interpolation="nearest")
    for b in bnd:
        ax.axvline(b, color="#66ccff", lw=0.4, alpha=0.55)
        ax.axhline(b, color="#66ccff", lw=0.4, alpha=0.55)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xlabel("time (s)")
    if ax is axes[0]:
        ax.set_ylabel("time (s)")
fig.suptitle(f"data2vec audio base - frame-by-frame self-similarity (blue lines = PHN boundaries)", y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
sep, keep, yk = phone_separability(hs, frame_classes, min_count=5)
display(sep.style.format({"silhouette_cos": "{:+.4f}", "probe_acc_5fold": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot(sep.layer, sep.silhouette_cos, "o-", color="#B279A2")
axes[0].axhline(0, ls="--", c="gray", lw=1)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("silhouette (cosine)")
axes[0].set_title("cluster geometry of broad phone classes"); axes[0].grid(alpha=0.3)
axes[1].plot(sep.layer, sep.probe_acc_5fold, "o-", color="#F58518")
axes[1].axhline(pd.Series(yk).value_counts(normalize=True).max(), ls="--", c="gray", lw=1,
                label="majority-class baseline")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("5-fold accuracy")
axes[1].set_title("linear probe (broad phone class)"); axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle(f"data2vec audio base - per-layer phone separability "
             f"(ONE utterance, {keep.sum()} frames - very noisy)", y=1.05)
fig.tight_layout(); plt.show()

print(f"frames used: {keep.sum()} / {T}, classes: {sorted(set(yk))}")
print(f"best silhouette: L{int(sep.silhouette_cos.idxmax())}   "
      f"best probe: L{int(sep.probe_acc_5fold.idxmax())}")
print("""
WARNING: with a single utterance the layer ordering of these two curves is NOT reliable and
should not be compared against published results. To get a meaningful curve, wrap
forward_frozen over a few hundred utterances, accumulate per-frame features and labels until
you have ~1e5 frames, and split train/test by speaker.
""")

In [ ]:
# PCA: project each frame to 2-D, colour by broad phone class
from sklearn.decomposition import PCA

show = [0, 4, 8, 12]
fig, axes = plt.subplots(1, len(show), figsize=(4.1 * len(show), 4.0))
for ax, li in zip(axes, show):
    Z = PCA(n_components=2, random_state=0).fit_transform(hs[li][0].float().numpy())
    for c in sorted(set(frame_classes)):
        m = frame_classes == c
        ax.scatter(Z[m, 0], Z[m, 1], s=20, c=CLASS_COLOR[c], label=c,
                   edgecolors="none", alpha=0.85)
    ax.set_title(f"hidden_states[{li}]", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(fontsize=7, loc="best", frameon=False)
fig.suptitle(f"data2vec audio base - PCA of per-frame representations (colour = broad phone class)", y=1.03)
fig.tight_layout(); plt.show()